# Macaron dish drawings
Draws every dish in the chosen cute style with Z-Image Turbo, on both GPUs at once, cuts out the background and saves small webp pictures plus contact sheets for checking. Everything is generated, so no one else's photos are used.

In [ ]:
!pip -q install -U --no-deps diffusers 2>&1 | tail -1
!pip -q install -U bitsandbytes 2>&1 | tail -1
!pip -q install --no-deps rembg pymatting onnxruntime flatbuffers coloredlogs humanfriendly 2>&1 | tail -2
import torch; print("GPUs:", torch.cuda.device_count())

In [ ]:
# which dishes to draw in this run, and the style
ONLY = [139, 525, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020, 3021, 3022, 3023, 3024, 3025, 3026, 3027, 3028, 3029, 3030, 3031, 3032, 3033, 3034, 3035, 4001, 4002, 4003, 4004, 4005, 4006, 4007, 4008, 4009, 4010, 4011, 4012, 4013, 4014, 4015, 4016, 4017, 4018, 4019, 4020, 4021, 4022, 4023, 4024, 4025, 4026, 4027, 4028, 4029, 4030, 4031, 4032, 4033, 4034, 4035, 4036, 4037, 4038]
STYLE_KEY = "Sticker"
SIZE, STEPS = 512, 8


In [ ]:
PROMPTS = [{"id": 0, "name": "Nasi lemak biasa", "desc": "Malaysian nasi lemak on a piece of banana leaf. A dome of white coconut rice, a spoon of dark red sambal, a small heap of crispy fried anchovies, a small heap of roasted peanuts, four cucumber slices and half a hard-boiled egg showing the yellow yolk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 9}, {"id": 76, "name": "Chicken rice, steamed", "desc": "a plate of Malaysian Hainanese steamed chicken rice. Six neat slices of boneless poached chicken with smooth pale white skin, laid in a row on cucumber slices, drizzled with light soy sauce and sprinkled with spring onion and coriander. A smooth dome of pale yellow rice beside it and a tiny saucer of red chilli sauce", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 82, "name": "Chicken rice, roasted", "desc": "a plate of Malaysian roasted chicken rice. Six neat boneless slices of roasted chicken, each slice with a thin layer of glossy dark golden-brown roasted skin on top and juicy white meat underneath, laid in a row on cucumber slices with a little soy sauce. A smooth dome of pale yellow chicken rice beside it and a tiny saucer of red chilli sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 5, "name": "Roti bawang", "desc": "a plate of Malaysian roti bawang. One round flat golden-brown pan-fried flatbread with crisp darker spots, its top generously covered with lots of chopped purple-red onion pieces pressed into the bread and lightly browned, clearly showing many onion bits. Beside it, a small bowl of orange-yellow dhal curry", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 10}, {"id": 141, "name": "Curry laksa", "desc": "a bowl of Malaysian curry laksa. Orange coconut curry soup with yellow noodles, three tofu puffs, four prawns, half a boiled egg, bean sprouts and fresh mint leaves", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 135, "name": "Char kway teow", "desc": "a plate of Penang char kuey teow. A loose pile of wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands, stir-fried and stained glossy brown with dark soy sauce, with slightly charred edges. Mixed in are three pink prawns, small cockles, crunchy bean sprouts, short green chive pieces and small yellow bits of scrambled egg (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 51, "name": "Wantan mee, dry", "desc": "a plate of Malaysian dry wantan mee. A heap of thin springy egg noodles tossed in glossy dark soy sauce, topped with five thin slices of red-edged char siu barbecued pork and a few stalks of green choy sum. Beside the plate, a small bowl of clear soup with three silky boiled wontons", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 119, "name": "Mee goreng mamak", "desc": "a plate of Malaysian mee goreng mamak. A heap of yellow noodles stir-fried until light brown and dry, with no sauce or gravy, mixed with cubes of fried tofu, small potato pieces, bean sprouts and green leafy vegetables, with no meat, topped with one fried egg with a runny yellow yolk, and a lime half on the side", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 10}, {"id": 103, "name": "Nasi goreng kampung", "desc": "Malaysian village fried rice with anchovies and water spinach (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 133, "name": "Satay ayam", "desc": "Malaysian chicken satay. Six bamboo skewers of grilled chicken pieces with caramelised charred edges, a small bowl of thick peanut sauce, cubes of cucumber and red onion, and a few rice cake cubes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 13, "name": "Kaya toast set", "desc": "Malaysian kaya toast set. Two thin slices of crisp toasted bread cut into rectangles, sandwiched with a layer of green kaya jam and a thick cold slice of yellow butter. Beside them, a small white bowl holding two runny soft-boiled eggs, cracked open, with dark soy sauce. A cup of black kopi coffee", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 610, "name": "Teh C", "desc": "a cup of hot Malaysian teh C on a saucer. A thick white porcelain kopitiam cup filled with smooth light tan-brown tea mixed with evaporated milk, a little steam rising, sitting on a matching white saucer with a small metal spoon, a plain cup with no decorations", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 6}, {"id": 1091, "name": "Fried wantan", "desc": "a small plate of five golden deep-fried wontons. Each wonton is a round puffy pouch with crinkly, ruffled, frilly crispy edges like a flower. A small dish of red sweet chilli sauce beside them", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 935, "name": "Curry puff", "desc": "Malaysian curry puffs: golden half-moon pastries with crimped twisted edges (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 788, "name": "PAUL croissant", "desc": "croissant (1 croissant)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 9}, {"id": 775, "name": "Cendol", "desc": "a bowl of Malaysian cendol. A mound of white shaved ice piled high in the middle of a bowl, drizzled with dark brown palm sugar syrup and white coconut milk running down the sides, with a heap of bright green worm-shaped pandan jelly strands on one side and a heap of glossy dark red kidney beans on the other side", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 857, "name": "Ice cream", "desc": "a single scoop of vanilla ice cream in a small paper cup. One round scoop of creamy white ice cream with a rough scooped texture, clear curved ridges and ripples left by the ice cream scoop, a little curled lip of ice cream around its base and a soft melting drip, sitting in a short pastel paper cup with a tiny wooden spoon stuck in it", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 14}, {"id": 148, "name": "Prawn mee", "desc": "Penang prawn mee: red-orange prawn broth with yellow noodles and vermicelli, prawns, pork slices, half a boiled egg, water spinach and fried shallots (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 56, "name": "Pan mee, soup", "desc": "a bowl of Malaysian pan mee soup. Hand-torn flat irregular pieces of pale noodle in clear light brown broth, topped with a generous heap of dark brown fried minced pork, sliced brown shiitake mushrooms, a pile of small crispy golden-brown fried anchovies (ikan bilis) clearly visible on top, and a bunch of very dark deep green leafy vegetables", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 95, "name": "Economy rice: 1 meat, 2 veg", "desc": "Malaysian economy rice: white rice with a piece of meat and two heaps of stir-fried vegetables, with : 1 meat, 2 veg (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 153, "name": "Bak kut teh with rice", "desc": "a claypot of Malaysian herbal bak kut teh soup with a small bowl of rice beside it. A round dark brown clay pot filled with dark brown herbal soup, with four chunky pork ribs on the bone, a whole bulb of garlic, brown shiitake mushrooms and fried tofu puffs, with Chinese herbs floating in the soup: bright red goji berries, a brown star anise, thin pale slices of dried angelica root and a stick of cinnamon bark. Beside it, a small bowl of plain white rice and a small dish of chopped red chilli in soy sauce", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 139, "name": "Hokkien mee (KL)", "desc": "a plate of KL Hokkien mee, also called black mee. A heap of thick noodles completely coated and soaked in very dark, glossy, pitch black thick soy sauce, so every noodle is black and shiny with no yellow showing anywhere, mixed with dark leaves of cabbage, a few pink prawns, slices of pork and golden crispy cubes of fried pork lard scattered on top", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 152, "name": "Yong tau foo, soup", "desc": "a bowl of Malaysian yong tau foo soup. Clear pale broth with white round fish balls, golden fried tofu puffs stuffed with white fish paste, a chunk of purple eggplant stuffed with fish paste, green okra lady fingers stuffed with fish paste, and a long red chilli stuffed with fish paste, sprinkled with chopped spring onion. Beside it, a small dish of red chilli sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 20, "name": "Chicken porridge", "desc": "a bowl of Chinese chicken porridge. Smooth thick white rice congee, topped with neat slices of pale poached chicken laid in a fan like Hainanese chicken rice, sprinkled with chopped green spring onion, golden fried shallots and thin ginger strips, with a drizzle of soy sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 833, "name": "Pho", "desc": "a bowl of Vietnamese beef pho. Clear light brown broth with flat white rice noodles, covered with many very thin wide slices of cooked dark brown beef, a big handful of white bean sprouts, fresh Thai basil leaves with purple stems, long green sawtooth coriander leaves, sliced white onion and a lime wedge on the rim", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 597, "name": "Latte, iced", "desc": "a tall glass of iced latte. A clear tall glass filled with ice cubes and cold white milk, with dark brown espresso swirling down from the top into the milk making soft brown and white layers, and a paper straw, a plain glass with no decorations", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 35}, {"id": 609, "name": "Kopi C", "desc": "a kopitiam cup of strong Malaysian coffee (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 9}, {"id": 617, "name": "Ticco milk tea", "desc": "a plastic cup of milk tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 14}, {"id": 947, "name": "Milo ais", "desc": "a glass of chocolate malt drink (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 396, "name": "Nasi Lemak McD", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 9, "name": "Roti telur", "desc": "a plate of Malaysian roti telur. One thin flat square folded flatbread, pan-fried golden brown with darker crisp spots, slightly puffed, with thin yellow egg visible at a torn edge, lying flat on a round steel plate. Beside it, a small bowl of orange-yellow dhal curry", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 10, "name": "Thosai with dhal and chutney", "desc": "a plate of South Indian thosai. A thin large golden-brown crispy crepe folded in half on a round steel plate. Beside it, a small bowl of yellow dhal curry with chunks of potato and a few dark green curry leaves floating in it, and a small bowl of white coconut chutney", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 115, "name": "Nasi briyani ayam", "desc": "Malaysian nasi briyani: long-grain yellow and orange spiced rice with a chicken curry, papadam and acar, with ayam (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 522, "name": "Banana leaf rice with chicken", "desc": "a South Indian banana leaf rice meal with chicken. A very large rectangular bright green banana leaf used as the plate, filling most of the picture, with a mound of white rice in the centre with curry ladled over it, three small heaps of vegetable side dishes along the top of the leaf (yellow cabbage, green beans, orange pumpkin), a piece of fried chicken, and two round crispy papadam crackers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 484, "name": "Char siu rice", "desc": "a plate of Malaysian char siu rice. A mound of white rice with six thick slices of char siu barbecued pork fanned on top, each slice with a dark red caramelised charred edge, a glossy sticky glaze and pink-brown meat with a thin strip of fat, drizzled with dark brown sweet char siu sauce that runs onto the rice, with three cucumber slices on the side", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 991, "name": "Roast duck rice", "desc": "chopped roast duck pieces with dark glossy skin in a little sauce, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 158, "name": "Steamed fish with rice and greens", "desc": "a plate of Chinese steamed fish with rice. A whole steamed white fish with pale silvery grey skin and soft white flesh, lying in a shallow pool of thin light soy sauce, topped with thin strips of ginger, spring onion and red chilli. Beside it, one small bowl of white rice", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 287, "name": "Tom yum soup with rice", "desc": "a bowl of Thai tom yum soup with a separate bowl of rice. A bowl of bright orange-red spicy clear tom yum soup with whole pink prawns, straw mushrooms, tomato wedges, lemongrass stalks, lime leaves and fresh coriander. Beside it, a small separate bowl of plain white steamed rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 297, "name": "Pad thai", "desc": "Thai pad thai: stir-fried rice noodles with prawns, egg, bean sprouts, crushed peanuts, chives and a lime wedge (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 525, "name": "Bankara ramen", "desc": "a bowl of Japanese tonkotsu ramen with a pair of chopsticks lifting a big bundle of long thin wavy yellow ramen noodles up out of the creamy light brown pork broth, so the noodles are the main thing you see, with two round slices of chashu pork, a soft-boiled egg cut in half showing a jammy orange yolk, a sheet of dark nori and chopped green spring onion", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 237, "name": "Sushi plates", "desc": "Japanese Sushi plates (6 pieces nigiri and maki, edamame)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 266, "name": "Chicken katsu curry rice", "desc": "a plate of Japanese chicken katsu curry rice. A mound of white rice on one side, a golden breaded fried chicken cutlet sliced into strips laid across the middle, and thick glossy brown Japanese curry sauce on the other side with big chunks of yellow potato and bright orange carrot in the sauce, with a little red pickled ginger", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 217, "name": "Cheeseburger", "desc": "a cheeseburger. A soft round golden sesame bun, a brown beef patty, a slice of melted yellow cheese, green lettuce, a red tomato slice and pickles, stacked neatly", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 317, "name": "Pizza", "desc": "a whole round pizza cut into eight slices. A thick puffy golden-brown baked crust with darker toasted spots, red tomato sauce, a generous layer of melted bubbly mozzarella cheese with browned patches, fully covered with plenty of toppings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 16}, {"id": 397, "name": "French fries", "desc": "French fries", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 410, "name": "KFC Original Recipe, thigh", "desc": "a single piece of crispy fried chicken thigh on a small plate. A chunky irregular piece of chicken with a thick craggy, crunchy, deep golden-brown breaded coating full of crispy ridges and flakes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 822, "name": "Egg tart", "desc": "Hong Kong egg tarts with smooth yellow custard in flaky pastry (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 10}, {"id": 572, "name": "Aseana nasi lemak", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 plate with chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 23}, {"id": 351, "name": "Mala mini hotpot", "desc": "Chinese hotpot (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 15}, {"id": 814, "name": "Christine's bun", "desc": "bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 13}, {"id": 219, "name": "Acme burger", "desc": "burger (With fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 11}, {"id": 779, "name": "MET fruit tea", "desc": "fruit tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 11}, {"id": 803, "name": "White coffee", "desc": "a cup of Ipoh white coffee with light brown froth (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 10}, {"id": 169, "name": "Fish and chips", "desc": "Fish and chips (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 9}, {"id": 624, "name": "MET matcha latte", "desc": "matcha latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 9}, {"id": 17, "name": "Dolly Dim Sum har gow", "desc": "three translucent har gow shrimp dumplings in a bamboo steamer; dim sum in bamboo steamers: har gow, siew mai and a steamed bun (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 417, "name": "Popcorn chicken", "desc": "Popcorn chicken", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 692, "name": "Cadbury Dairy Milk", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 802, "name": "Mango Magic smoothie", "desc": "Mango Magic smoothie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 837, "name": "TWG macarons", "desc": "macarons (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 50, "name": "Ali Muthu roti bakar set", "desc": "toasted bread with butter and kaya (Toast, 2 eggs and kopi)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 79, "name": "Yee3 chicken rice", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 276, "name": "Bibimbap", "desc": "Korean Bibimbap (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 325, "name": "ABC One fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 353, "name": "Da On galbi set", "desc": "Korean galbi set", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 811, "name": "Har gow", "desc": "three translucent har gow shrimp dumplings in a bamboo steamer (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 167, "name": "Dave's chicken chop", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw (With fries and coleslaw)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 184, "name": "Marco carbonara", "desc": "carbonara (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 343, "name": "Pumpkin soup", "desc": "Pumpkin soup", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 358, "name": "Palsaik kimchi stew", "desc": "Korean kimchi stew (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 361, "name": "Tteokbokki", "desc": "Korean Tteokbokki (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 602, "name": "Mocha, iced", "desc": "Mocha, iced", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 611, "name": "Wong Kok milk tea, iced", "desc": "a plastic cup of milk tea, with iced (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 6}, {"id": 780, "name": "Fong Lye beef noodles", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 853, "name": "Doughnut", "desc": "Doughnut (1 glazed)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 854, "name": "Butter cake", "desc": "Butter cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 84, "name": "Fong Lye pork chop rice", "desc": "Chinese pork chop rice (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 91, "name": "Yuan Room roast duck", "desc": "chopped roast duck pieces with dark glossy skin in a little sauce (Quarter duck)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 98, "name": "Bumbu Desa nasi campur", "desc": "Malaysian mixed rice: white rice with several small heaps of dishes such as curry chicken, vegetables and fried fish (1 plate, 3 dishes)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 144, "name": "KAKI 5 curry mee", "desc": "Malaysian curry mee: orange coconut curry soup with yellow noodles, tofu puffs, long beans and cockles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 166, "name": "Dandy grilled chicken plate", "desc": "grilled chicken plate (With rice and salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 211, "name": "Nando's chicken burger", "desc": "chicken burger (Breast fillet, peri-peri)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 232, "name": "DayOne salmon don", "desc": "Japanese salmon don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 238, "name": "Salmon nigiri", "desc": "Japanese Salmon nigiri (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 302, "name": "Pineapple fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with pineapple (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 318, "name": "Pizza, BBQ chicken", "desc": "Pizza, BBQ chicken", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 333, "name": "Nyonya chap chye with rice", "desc": "Chinese Nyonya chap chye with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 427, "name": "Tonkatsu set", "desc": "Japanese Tonkatsu set (cutlet, rice, cabbage, miso soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 573, "name": "Ayam gepuk with rice", "desc": "Malaysian with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 605, "name": "Lindt hot chocolate", "desc": "hot chocolate (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 706, "name": "Oreo Mini", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 798, "name": "Chocolate cake", "desc": "Chocolate cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 804, "name": "Dave's ribeye steak", "desc": "ribeye steak", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 5}, {"id": 841, "name": "Egg and cheese bagel", "desc": "Egg and cheese bagel (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 868, "name": "Croffle", "desc": "Korean Croffle (1 croissant waffle with syrup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 63, "name": "Mee hoon soup with fish balls", "desc": "white round fish balls; thin rice vermicelli, with soup with s (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 65, "name": "Imbi pork noodles, soup", "desc": "Chinese pork noodles, soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 107, "name": "ParaThai pad kra pao chicken", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg, with chicken (With rice and egg)", "neg": "whole roast chicken", "n": 4}, {"id": 122, "name": "Food court char kuey teow", "desc": "a plate of Penang char kuey teow. A loose pile of wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands, stir-fried and stained glossy brown with dark soy sauce, with slightly charred edges. Mixed in are three pink prawns, small cockles, crunchy bean sprouts, short green chive pieces and small yellow bits of scrambled egg, with food court (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 162, "name": "Johnny's chicken chop", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw (With fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 164, "name": "ABC One chicken chop rice", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 165, "name": "Baked pork chop rice", "desc": "Hong Kong baked pork chop rice: rice covered with fried pork chop and tomato sauce, baked with cheese in an oval dish (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 195, "name": "Quinoa chicken bowl", "desc": "Quinoa chicken bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 196, "name": "LAC protein bowl", "desc": "protein bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 220, "name": "Dandy beef burger", "desc": "beef burger (With fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 240, "name": "Kinori salmon nori taco", "desc": "Japanese salmon nori taco (1 taco)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 262, "name": "Ramen, shoyu", "desc": "Japanese Ramen, shoyu (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 267, "name": "Sukiya curry rice", "desc": "Japanese curry rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 279, "name": "bhc Bburinkle chicken", "desc": "Korean Bburinkle chicken (Half chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 280, "name": "Kyochon Soy Garlic wings", "desc": "Korean Soy Garlic wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 307, "name": "Chicken shawarma wrap", "desc": "Middle Eastern Chicken shawarma wrap (1 wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 332, "name": "Kangkung belacan with tofu and rice", "desc": "Malaysian Kangkung belacan with tofu and rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 349, "name": "Cucur udang", "desc": "cucur udang: round golden prawn fritters with chilli sauce (5 pieces with chili sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 364, "name": "K Fry kimchi fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with kimchi (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 411, "name": "A&W chicken, 2 pieces", "desc": "chicken, 2 pieces (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 424, "name": "Hot dog", "desc": "Hot dog (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 506, "name": "Ayam bakar with rice", "desc": "Malaysian bakar with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 604, "name": "Tim Hortons Iced Capp", "desc": "Iced Capp", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 686, "name": "KitKat Chunky", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 714, "name": "Chocolate chip cookie", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 743, "name": "Ritz crackers", "desc": "round crackers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 796, "name": "Fruit tart", "desc": "Fruit tart", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 830, "name": "Matcha tiramisu", "desc": "Japanese Matcha tiramisu (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 851, "name": "adough donut", "desc": "donut (1 donut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 871, "name": "Pretzel, original", "desc": "Pretzel, original (1 pretzel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 878, "name": "Eureka popcorn", "desc": "Eureka popcorn", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 897, "name": "Black bean soup with chicken", "desc": "Chinese Black bean soup with chicken", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 7, "name": "Mantra butter chicken with naan", "desc": "creamy orange butter chicken curry with a piece of naan; a soft blistered naan flatbread (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 22, "name": "Hon Kee fish porridge", "desc": "smooth Cantonese rice porridge, with fish (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 27, "name": "Oats with milk and banana", "desc": "Oats with milk and banana (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 36, "name": "Pokok big breakfast", "desc": "big breakfast (Eggs, sausage, toast and salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 53, "name": "On Lee wonton noodles", "desc": "thin springy egg noodles with slices of char siu and small boiled wontons (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 3}, {"id": 55, "name": "PappaRich wantan mee", "desc": "Malaysian wantan mee: thin springy egg noodles tossed in dark soy sauce with slices of red-edged char siu, choy sum and small boiled wontons (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 3}, {"id": 60, "name": "KAKI 5 pan mee", "desc": "pan mee: hand-torn flat noodles in clear broth with minced pork, fried anchovies, mushrooms and sweet potato leaves (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 64, "name": "Hot and sour soup", "desc": "Chinese Hot and sour soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 89, "name": "Village roast duck rice", "desc": "Chinese rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 99, "name": "Food hall mixed rice", "desc": "Malaysian mixed rice: white rice with a piece of meat and two heaps of stir-fried vegetables, with food hall (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 145, "name": "Grandmama's asam laksa", "desc": "Penang asam laksa: thick white rice noodles in brown sour fish broth topped with shredded cucumber, onion, pineapple, mint and a spoon of black prawn paste (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 146, "name": "Madam Kwan's assam laksa", "desc": "Penang asam laksa: thick white rice noodles in brown sour fish broth topped with shredded cucumber, onion, pineapple, mint and a spoon of black prawn paste (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 154, "name": "Klang bak kut teh", "desc": "Malaysian bak kut teh: a clay pot of dark herbal pork rib soup with pork ribs, tofu puffs, mushrooms and lettuce, with klang", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 156, "name": "Claypot chicken rice", "desc": "a black clay pot of rice topped with chicken pieces, Chinese sausage slices and spring onion (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 160, "name": "Crispy popiah", "desc": "small crispy fried popiah spring rolls (4 rolls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 168, "name": "Nando's macho peas", "desc": "macho peas", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 173, "name": "Sekinchan grilled fish rice", "desc": "grilled fish rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 189, "name": "Come True cafe brunch pasta", "desc": "cafe brunch pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 193, "name": "Spaghetti carbonara", "desc": "Spaghetti carbonara (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 231, "name": "Rakuzen chirashi don", "desc": "Japanese chirashi don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 247, "name": "Chicken teriyaki don", "desc": "Japanese Chicken teriyaki don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 249, "name": "Teppanyaki salmon set", "desc": "Japanese salmon set (With rice and vegetables)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 252, "name": "DayOne teriyaki chicken rice", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with teriyaki (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 263, "name": "Ippudo chicken ramen", "desc": "Japanese chicken ramen (1 bowl, pork-free)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 270, "name": "Itsu chicken katsu don", "desc": "Japanese chicken katsu don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 303, "name": "Johnny's Thai fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with thai (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 327, "name": "Wok Hey egg fried rice with chicken", "desc": "golden egg fried rice with spring onion and diced vegetables, with egg with chicken (1 box)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 337, "name": "Putien stir-fried yam", "desc": "Chinese stir-fried yam", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 342, "name": "Watercress soup with lean pork", "desc": "Chinese Watercress soup with lean pork", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 370, "name": "Kyochon Red Original wings", "desc": "Korean Red Original wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 382, "name": "Al Halabi mixed grill", "desc": "Middle Eastern mixed grill (With rice and salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 443, "name": "Thai Thai green curry chicken", "desc": "Malaysian chicken curry with potatoes in orange gravy, with thai thai green (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 477, "name": "Chili's chicken fajitas", "desc": "chicken fajitas (1 skillet with tortillas)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 478, "name": "Pho Vietz fresh spring rolls", "desc": "golden crispy fried spring rolls, with fresh s (2 rolls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 507, "name": "Ayam pongteh with rice", "desc": "Malaysian Ayam pongteh with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 509, "name": "Mantra chicken biryani", "desc": "long-grain yellow and orange spiced biryani rice with fried shallots, with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 518, "name": "Roti bom", "desc": "a small thick round flaky roti bom with sugar and butter (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 520, "name": "Rava thosai", "desc": "a thin crispy golden dosa folded on a plate with coconut chutney and sambar, with rava (1 piece with chutney)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 526, "name": "Bankara tsukemen", "desc": "Japanese Bankara tsukemen (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 576, "name": "Warung Prima nasi kunyit with chicken", "desc": "Malaysian nasi kunyit with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 592, "name": "Salmon sashimi", "desc": "Japanese Salmon sashimi (5 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 613, "name": "Taimal herbal tea", "desc": "Chinese herbal tea (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 620, "name": "HEYTEA cheese grape", "desc": "Chinese cheese grape", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 621, "name": "Leekor Leekor drink", "desc": "drink", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 691, "name": "KitKat White", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 694, "name": "Tim Tam Original", "desc": "Tim Tam Original", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 695, "name": "Chocolate bar", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 696, "name": "Twix", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 725, "name": "Cheezels", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 792, "name": "Cinnabon MiniBon", "desc": "MiniBon (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 793, "name": "Dome chicken pie", "desc": "chicken pie (1 pie with salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 801, "name": "Boost Juice Berry Crush", "desc": "Juice Berry Crush (Original size)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 818, "name": "Pancakes with syrup", "desc": "Pancakes with syrup (3 pancakes)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 847, "name": "Godiva chocolixir", "desc": "chocolixir", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 862, "name": "Banana pudding", "desc": "Banana pudding (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 870, "name": "Soft drink", "desc": "a glass of fizzy drink with ice", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 872, "name": "Cheesecake", "desc": "Cheesecake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 891, "name": "Chicken soup noodles", "desc": "Chinese Chicken soup noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 895, "name": "ABC soup with pork ribs", "desc": "Chinese ABC soup with pork ribs", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 1, "name": "Bungkus Kaw Kaw nasi lemak sotong", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with sotong (1 packet)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 2, "name": "Lele penyet with rice", "desc": "Malaysian Lele penyet with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 21, "name": "Canton-i century egg and pork congee", "desc": "smooth Cantonese rice porridge, with century egg and pork (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 26, "name": "Egg McMuffin", "desc": "Egg McMuffin (1 muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 28, "name": "Greek yogurt with fruit and oats", "desc": "Greek yogurt with fruit and oats (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 31, "name": "Egg sandwich", "desc": "Egg sandwich (2 slices wholemeal, 2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 32, "name": "LOVNS banh mi", "desc": "Vietnamese banh mi (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 35, "name": "PAUL croque monsieur", "desc": "croque monsieur (1 sandwich with salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 38, "name": "6-inch Subway Melt", "desc": "6-inch Subway Melt (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 43, "name": "O'Briens chicken sandwich", "desc": "chicken sandwich (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 45, "name": "Egg mayo sandwich", "desc": "Egg mayo sandwich (1 pack)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 57, "name": "AK Noodles pan mee, dry", "desc": "pan mee: hand-torn flat noodles in clear broth with minced pork, fried anchovies, mushrooms and sweet potato leaves, with noodles , dry (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 58, "name": "AK Noodles chili pan mee", "desc": "dry pan mee noodles with minced pork, crispy fried anchovies, a poached egg and dark chilli flakes on the side, with noodles (1 bowl, with poached egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 59, "name": "Dragon-i dan dan noodles", "desc": "Sichuan dan dan noodles with minced pork in red chilli oil and sesame sauce, with noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 62, "name": "Lor mee", "desc": "lor mee: yellow noodles in thick dark starchy gravy with braised pork, egg and fried fish (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 78, "name": "MyIPOH bean sprout chicken", "desc": "Chinese bean sprout chicken (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 80, "name": "Yun Nans steam pot chicken soup", "desc": "Chinese steam pot chicken soup (1 pot with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 83, "name": "Canton-i char siew rice", "desc": "rice with slices of red-edged glossy char siu barbecued pork and cucumber (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 123, "name": "Jjajangmyeon", "desc": "Korean jajangmyeon: thick noodles covered in glossy black bean sauce with diced pork and onion, with cucumber strips (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 142, "name": "Simple Life vegetarian laksa", "desc": "Malaysian laksa noodle soup, with vegetarian (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 150, "name": "Pu Tien lor mee", "desc": "lor mee: yellow noodles in thick dark starchy gravy with braised pork, egg and fried fish (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 175, "name": "Teochew steamed pomfret, per person", "desc": "Chinese Teochew steamed pomfret, per person (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 214, "name": "Texas Chicken sandwich", "desc": "Chicken sandwich (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 222, "name": "Double Whopper with cheese", "desc": "Double Whopper with cheese (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 235, "name": "Rakuzen bento set", "desc": "Japanese bento set (Sashimi, tempura and rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 242, "name": "NakNak bulgogi kimbap", "desc": "Korean bulgogi kimbap (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 248, "name": "Sushi Zanmai unagi don", "desc": "Japanese Sushi Zanmai unagi don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 250, "name": "Teppanyaki chicken set", "desc": "Japanese chicken set (With rice, bean sprouts and egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 253, "name": "Cloud 9 rice bowl", "desc": "Chinese rice bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 257, "name": "Eatomo chicken teriyaki bento", "desc": "Japanese chicken teriyaki bento (1 bento)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 261, "name": "Takeno udon set", "desc": "Japanese udon set (With tempura)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 278, "name": "K Fry soy garlic chicken", "desc": "Korean soy garlic chicken (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 282, "name": "Taiwanese XXL fried chicken", "desc": "Chinese Taiwanese XXL fried chicken (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 288, "name": "Thai Thai tom yum goong", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with thai thai goong (1 bowl with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 293, "name": "Andaman tom yum seafood soup", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with seafood soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 305, "name": "Boat noodles", "desc": "Thai noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 321, "name": "Gordon Ramsay Street Pizza pepperoni slice", "desc": "Pizza pepperoni slice (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 324, "name": "Steamed chicken, rice and broccoli", "desc": "Chinese Steamed chicken, rice and broccoli (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 334, "name": "Kita vegetarian rice bowl", "desc": "vegetarian rice bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 335, "name": "Cili Kampung sayur campur", "desc": "Malaysian sayur campur (1 side)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 344, "name": "Green papaya fish soup", "desc": "Chinese Green papaya fish soup", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 345, "name": "Seaweed and tofu soup", "desc": "Japanese Seaweed and tofu soup", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 354, "name": "Palsaik 8-flavour pork belly set, per person", "desc": "Korean 8-flavour pork belly set, per person (Pork belly, sides and rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 368, "name": "bhc Hot Fried chicken", "desc": "Korean Hot Fried chicken (Half chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 375, "name": "BBQ chicken wings", "desc": "BBQ chicken wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 376, "name": "Black Tap buffalo wings", "desc": "buffalo wings (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 384, "name": "India Gate tandoori chicken", "desc": "red tandoori chicken pieces with onion and lemon, with chicken (Half chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 388, "name": "Big Mac", "desc": "Big Mac (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 398, "name": "Shake Shack cheese fries", "desc": "cheese fries", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 399, "name": "Happy Potato tornado potato", "desc": "Korean tornado potato (1 stick)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 413, "name": "KFC Original Recipe, wing", "desc": "Original Recipe, wing (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 426, "name": "Ramly burger, biasa", "desc": "Malaysian Ramly burger, biasa (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 442, "name": "Green curry chicken with rice", "desc": "Malaysian chicken curry with potatoes in orange gravy, with green with rice (1 bowl curry, 1 cup rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 445, "name": "India Gate palak paneer", "desc": "Indian palak paneer (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 451, "name": "Tom yum noodles", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 453, "name": "Jjamppong", "desc": "Korean jjamppong: spicy red seafood noodle soup with mussels, squid and vegetables (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 455, "name": "Para Thai tom yum noodles", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with thai noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 459, "name": "Thai milk tea", "desc": "a plastic cup of milk tea, with thai", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 466, "name": "Kim Gary macaroni soup with ham", "desc": "Chinese Kim Gary macaroni soup with ham (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 476, "name": "Grilled chicken wrap", "desc": "Grilled chicken wrap (1 wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 497, "name": "Las Meninas tapas, per person", "desc": "tapas, per person", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 505, "name": "Ahh Yum nasi bunga telang", "desc": "Malaysian nasi bunga telang (1 plate with chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 535, "name": "Salted egg", "desc": "Chinese Salted egg (1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 584, "name": "Plain naan", "desc": "a soft blistered naan flatbread, with plain (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 598, "name": "Caramel macchiato, iced", "desc": "Caramel macchiato, iced", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 608, "name": "Kopi O", "desc": "a kopitiam cup of strong Malaysian coffee", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 612, "name": "Mr Bean soya milk", "desc": "Chinese soya milk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 614, "name": "Teh O", "desc": "a cup of black tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 630, "name": "Pho, chicken", "desc": "Vietnamese beef pho: clear broth, flat rice noodles, thin slices of beef, basil, bean sprouts and lime, with chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 631, "name": "MyIPOH Ipoh chicken hor fun", "desc": "smooth wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands, with ipoh chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 632, "name": "Ipoh hor fun", "desc": "Ipoh hor fun: smooth wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands in clear chicken broth with shredded chicken and prawns (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 633, "name": "Bun bo Hue", "desc": "Vietnamese bun bo hue: spicy red lemongrass beef noodle soup with thick round rice noodles and herbs (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 657, "name": "Omelette, plain", "desc": "Omelette, plain (2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 679, "name": "White rice", "desc": "White rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 685, "name": "Garlic bread", "desc": "Garlic bread (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 687, "name": "Mars", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 700, "name": "Pocky Strawberry", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 711, "name": "Munchy's Oat Krunch Chocolate", "desc": "oat cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 716, "name": "Dubai chewy cookie", "desc": "Dubai chewy cookie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 720, "name": "Kinder Joy", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 726, "name": "Mamee Monster", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 733, "name": "Hello Panda Strawberry", "desc": "small biscuits with chocolate dip", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 736, "name": "Twisties Tomato", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 751, "name": "Tora corn snack", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 768, "name": "Tic Tac", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 771, "name": "Mango lassi", "desc": "a glass of thick yellow mango lassi, with mango (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 782, "name": "Fong Lye beef noodle soup", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy, with soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 783, "name": "Rén la mian with minced pork", "desc": "Chinese hand-pulled noodles, with minced pork (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 784, "name": "Mama Yap Hakka noodles", "desc": "Chinese Hakka noodles (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 795, "name": "Almond croissant", "desc": "Almond croissant (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 797, "name": "Chateraise strawberry shortcake", "desc": "Japanese strawberry shortcake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 805, "name": "Shook! grilled wagyu", "desc": "grilled wagyu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 815, "name": "Ippudo pork bun", "desc": "Japanese pork bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 817, "name": "Char siu bao", "desc": "fluffy white steamed char siu bao buns split open at the top showing red barbecued pork (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 821, "name": "HWC chicken pie", "desc": "chicken pie (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 824, "name": "Komugi cheese tart", "desc": "Japanese cheese tart (1 tart)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 825, "name": "Gyoza", "desc": "pan-fried gyoza dumplings in a row with crispy browned bottoms (5 pieces, pan-fried)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 828, "name": "Injeolmi bingsu", "desc": "Korean Injeolmi bingsu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 829, "name": "Marco tiramisu", "desc": "tiramisu (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 831, "name": "Mee tarik beef soup", "desc": "Chinese beef soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 832, "name": "Pho Vietz beef pho", "desc": "Vietnamese beef pho: clear broth, flat rice noodles, thin slices of beef, basil, bean sprouts and lime, with beef (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 836, "name": "Soft serve cone", "desc": "Japanese Soft serve cone (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 842, "name": "Park's Bagels smoked salmon bagel", "desc": "smoked salmon bagel (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 845, "name": "Bagel, plain", "desc": "Bagel, plain (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 849, "name": "Chocolate milkshake", "desc": "Chocolate milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 850, "name": "Krispy Kreme chocolate iced glazed", "desc": "chocolate iced glazed (1 doughnut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 860, "name": "Inside Scoop, double scoop in a waffle cone", "desc": "double scoop in a waffle cone (2 scoops, 1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 865, "name": "Burnt cheesecake", "desc": "Burnt cheesecake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 866, "name": "Herbal Farmer herbal jelly", "desc": "Chinese herbal jelly (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 879, "name": "Scone with jam and cream", "desc": "Scone with jam and cream (1 scone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 880, "name": "Takoyaki", "desc": "Japanese Takoyaki (6 balls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 885, "name": "Pisang goreng", "desc": "Malaysian banana fritters: golden battered fried banana pieces (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 886, "name": "Ayam masak merah with rice", "desc": "chicken pieces in thick red tomato chilli sauce, with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 889, "name": "Pan-seared salmon, rice and greens", "desc": "Pan-seared salmon, rice and greens (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 900, "name": "Omelette with rice and cucumber", "desc": "Malaysian Omelette with rice and cucumber (2 eggs, 1 cup rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 3, "name": "Teh Tarik Nation nasi lemak", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg; a glass mug of Malaysian teh tarik milk tea, light brown with a thick frothy foam top, with nation (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 4, "name": "Nasi lemak telur rebus, kurang nasi", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with telur rebus, kurang nasi (Half rice, boiled egg, cucumber)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 6, "name": "Roti cheese", "desc": "Indian Roti cheese (1 piece with dhal)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 8, "name": "Teh Tarik Club roti canai with curry", "desc": "Malaysian roti canai: round flaky golden flatbread folded loosely on a plate with a small bowl of yellow dhal curry; a glass mug of Malaysian teh tarik milk tea, light brown with a thick frothy foam top, with club with curry (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 11, "name": "Chapati with dhal", "desc": "two soft round chapati flatbreads with dhal; small bowl of yellow lentil dhal curry (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 12, "name": "Idli with sambar", "desc": "three white round steamed idli rice cakes with sambar and chutney, with sambar (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 14, "name": "Half-boiled eggs with wholemeal toast", "desc": "a small bowl of two runny soft-boiled eggs with soy sauce and white pepper, with wholemeal toast (2 eggs, 1 slice toast, no butter)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 15, "name": "Chee cheong fun, plain", "desc": "plain rolled white rice noodle rolls drizzled with sweet brown sauce and sesame seeds, with plain (1 plate with sweet sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 16, "name": "Dim sum breakfast", "desc": "dim sum in bamboo steamers: har gow, siew mai and a steamed bun, with breakfast (3 har gow, 3 siew mai, 1 bao)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 18, "name": "Dolly Dim Sum chee cheong fun with prawns", "desc": "plain rolled white rice noodle rolls drizzled with sweet brown sauce and sesame seeds; dim sum in bamboo steamers: har gow, siew mai and a steamed bun, with prawns (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 19, "name": "Lor mai kai", "desc": "lor mai kai: glutinous rice with chicken and mushroom steamed in a small bowl (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 23, "name": "Souper Tang double-boiled chicken soup", "desc": "Chinese double-boiled chicken soup (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 24, "name": "Teochew porridge with sides", "desc": "smooth Cantonese rice porridge, with teochew with sides (1 bowl and 3 sides)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 25, "name": "Bubur Ayam McD", "desc": "rice porridge, with ayam mcd", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 29, "name": "Scrambled eggs on wholemeal toast", "desc": "Scrambled eggs on wholemeal toast (2 eggs, 2 slices)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 30, "name": "Peanut butter banana toast", "desc": "Peanut butter banana toast (2 slices wholemeal)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 33, "name": "Onigirazu, chicken katsu", "desc": "Japanese Onigirazu, chicken katsu (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 34, "name": "Onigirazu, salmon and egg", "desc": "Japanese Onigirazu, salmon and egg (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 37, "name": "Whisk eggs Benedict", "desc": "eggs Benedict (2 eggs on muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 39, "name": "6-inch Italian B.M.T. (chicken ham, pepperoni, salami)", "desc": "6-inch Italian B.M.T. (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 40, "name": "6-inch Chicken Slice sub", "desc": "6-inch Chicken Slice sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 41, "name": "6-inch Steak & Cheese sub", "desc": "6-inch Steak & Cheese sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 42, "name": "6-inch seafood sensation sub", "desc": "6-inch seafood sensation sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 44, "name": "Onigiri and boiled egg", "desc": "Japanese Onigiri and boiled egg (1 tuna onigiri, 1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 46, "name": "Paris Baguette ham and cheese sandwich", "desc": "ham and cheese sandwich (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 47, "name": "6-inch egg mayo sub", "desc": "6-inch egg mayo sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 48, "name": "Hainan toast with kaya, and eggs", "desc": "toasted bread with kaya and butter, with soft-boiled eggs; green kaya coconut jam, with and eggs (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 49, "name": "Steamed Hainan bread with kaya", "desc": "green kaya coconut jam, with steamed hainan bread (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 52, "name": "AK Noodles wantan mee", "desc": "Malaysian wantan mee: thin springy egg noodles tossed in dark soy sauce with slices of red-edged char siu, choy sum and small boiled wontons, with noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 54, "name": "Noodle Shack dry wantan noodles", "desc": "small boiled wontons with silky wrappers, with noodle shack dry noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 61, "name": "Mee tarik dry with minced beef", "desc": "Chinese dry with minced beef (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 66, "name": "Se'Indonesia bakso", "desc": "Malaysian bakso (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 67, "name": "Wanhoi prawn wantan soup", "desc": "small boiled wontons with silky wrappers, with prawn soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 68, "name": "Yunnan crossing-the-bridge noodles", "desc": "Yunnan crossing-the-bridge rice noodles in a big bowl of clear soup with toppings, with yunnan noodles (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 69, "name": "Yun Nans rice noodles with minced pork", "desc": "Chinese rice noodles with minced pork (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 70, "name": "Bone soup rice noodles", "desc": "Chinese Bone soup rice noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 71, "name": "Hot Crush pork noodles", "desc": "Chinese pork noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 72, "name": "Shihlin oyster mee sua", "desc": "Chinese oyster mee sua (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 73, "name": "Lontong", "desc": "lontong: rice cakes in yellow coconut vegetable gravy with egg and sambal (1 bowl with kuah)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 74, "name": "Nasi dagang", "desc": "Terengganu nasi dagang: reddish brown speckled rice with fish curry and pickles (1 plate with fish curry)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 75, "name": "Mee siam", "desc": "mee siam: thin pink rice vermicelli fried, with egg, tofu and lime (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 77, "name": "Bar B Q Plaza jasmine rice", "desc": "Thai jasmine rice (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 81, "name": "Thai chicken rice (khao man gai)", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with thai (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 85, "name": "Fong Lye Three-cup chicken set", "desc": "three cup chicken in a small claypot with basil leaves (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 86, "name": "Long Jing Dongpo pork with rice", "desc": "a glossy cube of dark braised pork belly, with rice (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 87, "name": "Oversea char siew", "desc": "slices of red-edged glossy char siu barbecued pork (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 88, "name": "Tai Ho Jiak lu rou fan", "desc": "Taiwanese braised minced pork over rice with a braised egg (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 90, "name": "Yat Yat Seng roast pork rice", "desc": "slices of crispy-skinned roast pork belly, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 92, "name": "Grand Imperial roast pork and duck rice", "desc": "slices of crispy-skinned roast pork belly, with duck rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 93, "name": "Peony roast meat rice", "desc": "Chinese roast meat rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 94, "name": "Steamed chicken rice set", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with steamed set (1 set with soup and bean sprouts)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 96, "name": "Economy rice: steamed fish, tofu, greens", "desc": "Malaysian economy rice: white rice with a piece of meat and two heaps of stir-fried vegetables, with : steamed fish, tofu, greens (Half rice)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 97, "name": "Esquire Kitchen sweet and sour fish with rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with fish with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 100, "name": "Nasi campur: ayam bakar, sayur, ulam", "desc": "Malaysian mixed rice: white rice with several small heaps of dishes such as curry chicken, vegetables and fried fish; grilled marinated chicken with charred edges; fresh Malay herbs and raw vegetables with sambal belacan, with : , sayur (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 101, "name": "Ikan bakar with rice and ulam", "desc": "fresh Malay herbs and raw vegetables with sambal belacan, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 102, "name": "Nasi ayam penyet", "desc": "Indonesian ayam penyet: smashed fried chicken with red sambal, rice, tofu, tempeh and cucumber, with nasi (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 104, "name": "Grandmama's nyonya fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with nyonya (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 105, "name": "Las Meninas seafood paella, per person", "desc": "seafood paella, per person (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 106, "name": "Nasi goreng with fried egg", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with fried egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 108, "name": "Black Canyon pad kra pao chicken rice", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce; Thai minced meat stir-fried with holy basil over rice with a fried egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 109, "name": "Jatujak pad kra pao with egg", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 110, "name": "Thai Hou Sek basil pork rice", "desc": "Thai Hou Sek basil pork rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 111, "name": "Chakri basil minced pork rice", "desc": "Thai basil minced pork rice (With egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 112, "name": "Pad kra pao with rice and egg", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg, with rice and egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 113, "name": "Pad kra pao chicken, home-style", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg, with chicken, home-style", "neg": "whole roast chicken", "n": 1}, {"id": 114, "name": "Nasi kandar", "desc": "Penang nasi kandar: white rice flooded with mixed dark curry gravies, fried chicken, okra and a boiled egg (Rice, fried chicken, mixed curry)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 116, "name": "Banana leaf rice, vegetarian", "desc": "South Indian banana leaf rice: white rice on a banana leaf with three small heaps of vegetable curries, a papadam and curry gravy, with rice, vegetarian (Rice, 3 vegetables, dhal, rasam)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 117, "name": "Tandoori chicken with naan", "desc": "red tandoori chicken pieces with onion and lemon; a soft blistered naan flatbread, with chicken (1 piece chicken, 1 plain naan)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 118, "name": "Barbeque Nation buffet, per person", "desc": "Indian buffet, per person (Grills, curries, rice and dessert)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 120, "name": "Vegetarian fried bee hoon", "desc": "thin rice vermicelli, with vegetarian fried (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 121, "name": "Esquire Kitchen golden pumpkin noodles", "desc": "Chinese golden pumpkin noodles (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 124, "name": "Mee tarik fried with chicken", "desc": "Chinese fried with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 125, "name": "Pu Tien Heng Hwa fried bee hoon", "desc": "thin rice vermicelli, with heng hwa fried (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 126, "name": "Wok Hey fried udon", "desc": "Chinese fried udon (1 box)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 127, "name": "Maggi goreng", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 128, "name": "Mee rebus", "desc": "mee rebus: yellow noodles in thick sweet potato gravy with boiled egg, tofu and lime (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 129, "name": "Mee bandung", "desc": "mee bandung: yellow noodles in thick red-orange prawn gravy with egg (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 130, "name": "Soto ayam", "desc": "soto ayam: clear yellow spiced chicken soup with shredded chicken, rice cubes and bean sprouts (1 bowl with nasi impit)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 131, "name": "Sup ayam with rice", "desc": "Malay chicken soup: clear spiced broth with chicken pieces, potato and carrot, with a bowl of rice, with rice (1 bowl soup, 1 cup rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 132, "name": "Laksam", "desc": "laksam: rolled white rice noodle sheets in thick creamy white fish gravy with herbs (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 134, "name": "Robataya grilled skewers, per person", "desc": "Japanese grilled skewers, per person (6 skewers)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 136, "name": "Putien Heng Hwa fried bee hoon", "desc": "thin rice vermicelli, with heng hwa fried (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 137, "name": "Kim Lian Kee Hokkien mee", "desc": "KL Hokkien mee: thick yellow noodles braised in glossy black soy sauce with pork, prawns, cabbage and crispy lard bits (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 138, "name": "Duck egg char kuey teow", "desc": "a plate of Penang char kuey teow. A loose pile of wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands, stir-fried and stained glossy brown with dark soy sauce, with slightly charred edges. Mixed in are three pink prawns, small cockles, crunchy bean sprouts, short green chive pieces and small yellow bits of scrambled egg, with duck egg (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 140, "name": "Wat tan hor", "desc": "wat tan hor: wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands in glossy white egg gravy with prawns and greens (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 143, "name": "Vegetarian curry mee", "desc": "Malaysian curry mee: orange coconut curry soup with yellow noodles, tofu puffs, long beans and cockles, with vegetarian (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 147, "name": "Asam laksa", "desc": "Penang asam laksa: thick white rice noodles in brown sour fish broth topped with shredded cucumber, onion, pineapple, mint and a spoon of black prawn paste (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 149, "name": "Mr Fish seafood noodles, milky broth", "desc": "Chinese seafood noodles, milky broth (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 151, "name": "Fish head mee hoon, clear soup", "desc": "fish head noodle soup: rice vermicelli in clear soup with fried fish pieces, tomato and salted vegetables, with clear soup (1 bowl, no milk)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 155, "name": "Pao Xiang dry bak kut teh", "desc": "a claypot of Malaysian dry bak kut teh. A round dark brown clay pot with no soup, filled with chunks of pork ribs and pork belly coated in a thick, sticky, very dark almost black soy sauce glaze, mixed with whole dried red chillies, sliced green okra lady fingers and strips of dried cuttlefish, glossy and caramelised", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 157, "name": "Kwan Kee claypot rice", "desc": "Chinese claypot rice (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 159, "name": "Sweet and sour chicken rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion; sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 161, "name": "Siwawa Guizhou wraps", "desc": "Chinese Siwawa Guizhou wraps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 163, "name": "Kim Gary baked cheese rice with chicken chop", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw, with kim gary baked cheese rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 170, "name": "Fish & Co Fish in a Pan", "desc": "Fish in a Pan (Fish fillets with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 171, "name": "Fish & Co seafood platter for one", "desc": "seafood platter for one (Fish, prawns, calamari and fries)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 172, "name": "Delirium Belgian mussels with fries", "desc": "Belgian mussels with fries (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 174, "name": "Laem Charoen deep-fried seabass with fish sauce", "desc": "Thai deep-fried seabass with fish sauce (Half fish)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 176, "name": "Peng Chu steamed fish with chili", "desc": "Chinese steamed fish with chili", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 177, "name": "Xu's Cuisine Hunan chili fish", "desc": "Chinese Hunan chili fish", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 178, "name": "De.Wan ikan bakar set", "desc": "grilled whole fish wrapped in charred banana leaf with red sambal (With rice and sambal)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 179, "name": "Robataya grilled saba set", "desc": "Japanese grilled saba set (With rice and miso soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 180, "name": "Steamed siakap with ginger and soy, rice and greens", "desc": "Chinese Steamed siakap with ginger and soy, rice and greens", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 181, "name": "Ikan kembung bakar with ulam and rice", "desc": "fresh Malay herbs and raw vegetables with sambal belacan, with ikan kembung bakar with and rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 182, "name": "Grilled fish with salad and sweet potato", "desc": "Grilled fish with salad and sweet potato (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 183, "name": "Steamed fish with tofu and spring onion, no rice", "desc": "Chinese Steamed fish with tofu and spring onion, no rice (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 185, "name": "Secret Recipe chicken lasagna", "desc": "chicken lasagna (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 186, "name": "Pizza Hut cheesy baked spaghetti, bolognese", "desc": "Pizza Hut cheesy baked spaghetti, bolognese (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 187, "name": "Spaghetti aglio olio with chicken", "desc": "Spaghetti aglio olio with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 188, "name": "ANTE pork belly aglio olio", "desc": "pork belly aglio olio (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 190, "name": "Ocean Twe12ve seafood pasta", "desc": "seafood pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 191, "name": "Aglio olio with prawns", "desc": "Aglio olio with prawns", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 192, "name": "Zucchini noodles with prawns and tomato", "desc": "Zucchini noodles with prawns and tomato (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 194, "name": "Kenny Rogers mac and cheese", "desc": "mac and cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 197, "name": "Se'Indonesia gado-gado", "desc": "Malaysian gado-gado (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 198, "name": "Daily Greens salmon salad", "desc": "Daily Greens salmon salad (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 199, "name": "Subway salad, roasted chicken", "desc": "salad, roasted chicken (No bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 200, "name": "Cauli & Rice grain bowl", "desc": "Rice grain bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 201, "name": "Lettuce wraps with minced chicken", "desc": "Chinese Lettuce wraps with minced chicken (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 202, "name": "Quarter chicken with 2 sides", "desc": "Quarter chicken with 2 sides (Chicken, garden salad, corn)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 203, "name": "Chicken pita", "desc": "Chicken pita (1 pita)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 204, "name": "Quarter chicken with 3 sides", "desc": "Quarter chicken with 3 sides (Chicken, coleslaw, vegetables, mash)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 205, "name": "6-inch roasted chicken sub", "desc": "6-inch roasted chicken sub (Wholemeal, all the veg, light sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 206, "name": "6-inch chicken teriyaki sub", "desc": "6-inch chicken teriyaki sub (Wholemeal, all the veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 207, "name": "6-inch tuna sub", "desc": "6-inch tuna sub (Wholemeal, all the veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 208, "name": "6-inch Veggie Delite with egg", "desc": "6-inch Veggie Delite with egg (Wholemeal, all the veg, egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 209, "name": "Filet-O-Fish", "desc": "Filet-O-Fish (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 210, "name": "McChicken", "desc": "McChicken (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 212, "name": "Shake Shack Chick'n Shack", "desc": "Chick'n Shack (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 213, "name": "Burger King fish burger", "desc": "Burger King fish burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 215, "name": "Marrybrown MB burger", "desc": "MB burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 216, "name": "Ayam Goreng McD", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 218, "name": "ANTE pork burger", "desc": "pork burger (1 burger with fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 221, "name": "Whopper Jr", "desc": "Whopper Jr (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 223, "name": "Original Recipe chicken, 2 pieces", "desc": "Original Recipe chicken, 2 pieces (With coleslaw)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 224, "name": "Zinger burger", "desc": "Zinger burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 225, "name": "Crispy chicken with rice", "desc": "Crispy chicken with rice (1 piece, rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 226, "name": "Chicken porridge (Marrybrown)", "desc": "smooth Cantonese rice porridge, with chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 227, "name": "Original chicken, 2 pieces", "desc": "Original chicken, 2 pieces (With mashed potato)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 228, "name": "Salmon poke bowl", "desc": "Japanese Salmon poke bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 229, "name": "Build-your-own salad bowl", "desc": "Build-your-own salad bowl (Greens, grilled chicken, egg, 3 vegetables)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 230, "name": "Salmon sashimi and rice", "desc": "Japanese Salmon sashimi and rice (6 slices, 1 bowl rice, miso soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 233, "name": "Fish Bowl poke bowl, salmon", "desc": "Japanese Fish Bowl poke bowl, salmon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 234, "name": "Fish Bowl poke bowl, tuna", "desc": "Japanese Fish Bowl poke bowl, tuna (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 236, "name": "Ichi Zen sashimi set", "desc": "Japanese sashimi set (With rice and miso soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 239, "name": "Empire Sushi California roll", "desc": "Japanese Sushi California roll (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 241, "name": "Makii Makii salmon hand roll", "desc": "Japanese salmon hand roll (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 243, "name": "Sushiro grilled eel nigiri", "desc": "Japanese grilled eel nigiri (1 plate of 2)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 244, "name": "Sushiro tuna nigiri", "desc": "Japanese tuna nigiri (1 plate of 2)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 245, "name": "Kinori spicy tuna nori taco", "desc": "Japanese spicy tuna nori taco (1 taco)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 246, "name": "Mama Kimbap tuna mayo kimbap", "desc": "Korean tuna mayo kimbap (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 251, "name": "Itsu salmon teriyaki bento", "desc": "Japanese salmon teriyaki bento (1 bento)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 254, "name": "Sukiya gyudon, regular", "desc": "Japanese gyudon, regular (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 255, "name": "Sukiya cheese gyudon", "desc": "Japanese cheese gyudon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 256, "name": "Kampachi gyu teriyaki set", "desc": "Japanese gyu teriyaki set (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 258, "name": "Tenkai tendon", "desc": "Japanese tendon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 259, "name": "Salmon teriyaki bento", "desc": "Japanese Salmon teriyaki bento (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 260, "name": "Udon in soup", "desc": "Japanese Udon in soup (1 bowl with egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 264, "name": "Menya Musashi tsukemen", "desc": "Japanese tsukemen (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 265, "name": "MyeongDong ramyeon", "desc": "Korean ramyeon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 268, "name": "Monster Curry pork katsu curry", "desc": "Japanese Curry pork katsu curry (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 269, "name": "Monster Curry chicken katsu curry", "desc": "Malaysian chicken curry with potatoes in orange gravy, with katsu curry (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 271, "name": "Ma Maison katsu curry", "desc": "Japanese Ma Maison katsu curry (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 272, "name": "Donkai katsu don", "desc": "Japanese katsu don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 273, "name": "DayOne katsu curry pork chop omelette rice", "desc": "Japanese katsu curry pork chop omelette rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 274, "name": "Gyudon", "desc": "Japanese Gyudon", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 275, "name": "Gyudon, mini with salad", "desc": "Japanese Gyudon, mini with salad (1 mini bowl, side salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 277, "name": "Miga beef bulgogi set", "desc": "Korean beef bulgogi set (With rice and sides)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 281, "name": "Kyochon Honey Original drumsticks", "desc": "Korean Honey Original drumsticks (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 283, "name": "MyeongDong Korean fried chicken", "desc": "Korean fried chicken (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 284, "name": "Soy garlic chicken rice box", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with soy garlic box (1 box)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 285, "name": "Kimbap", "desc": "Korean Kimbap (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 286, "name": "Japchae", "desc": "Korean Japchae (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 289, "name": "Cili Kampung asam pedas ikan with rice", "desc": "fish in a red sour spicy tamarind gravy with okra, with ikan with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 290, "name": "Laem Charoen tom yum seafood", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with seafood (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 291, "name": "Melaka Nyonya asam pedas fish with rice", "desc": "fish in a red sour spicy tamarind gravy with okra, with nyonya fish with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 292, "name": "Tai Er sauerkraut fish, per person", "desc": "Chinese sauerkraut fish, per person (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 294, "name": "Amarin tom yum goong", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with goong (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 295, "name": "Celadon tom yum goong", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with goong (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 296, "name": "Absolute Thai tom yum seafood", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with thai seafood (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 298, "name": "Absolute Thai pad thai with prawns", "desc": "Thai pad thai: stir-fried rice noodles with prawns, egg, bean sprouts, crushed peanuts, chives and a lime wedge, with thai with prawns (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 299, "name": "An Viet bun cha", "desc": "Vietnamese bun cha (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 300, "name": "Boat Noodle pad thai", "desc": "Thai pad thai: stir-fried rice noodles with prawns, egg, bean sprouts, crushed peanuts, chives and a lime wedge, with noodle (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 301, "name": "Thai basil chicken rice with egg", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce; Thai minced meat stir-fried with holy basil over rice with a fried egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 304, "name": "Laem Charoen crab fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with crab (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 306, "name": "Boat Noodle beef noodles", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy, with noodle s", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 308, "name": "Al Halabi hummus with bread", "desc": "Middle Eastern hummus with bread", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 309, "name": "Al-Amar chicken shawarma plate", "desc": "Middle Eastern chicken shawarma plate (With rice and garlic sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 310, "name": "Nasi mandi ayam", "desc": "Arabic nasi mandi: long-grain golden rice with a roasted chicken quarter on top and tomato salsa, with ayam (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 311, "name": "Nasi mandi ayam, half rice", "desc": "Arabic nasi mandi: long-grain golden rice with a roasted chicken quarter on top and tomato salsa, with ayam, half rice (Quarter chicken, half rice, salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 312, "name": "Chicken kebab plate with salad", "desc": "Middle Eastern Chicken kebab plate with salad (2 skewers, salad, 1 pita)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 313, "name": "Hummus and falafel plate", "desc": "Middle Eastern Hummus and falafel plate (Hummus, 4 falafel, 1 pita, salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 314, "name": "Lentil soup with pita", "desc": "Middle Eastern Lentil soup with pita (1 bowl, 1 pita)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 315, "name": "Kofta plate with rice", "desc": "Middle Eastern Kofta plate with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 316, "name": "Mi xian in soup with meatballs", "desc": "Yunnan rice noodle soup with meatballs and greens, with in soup with meatballs (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 319, "name": "Pizza, seafood (tuna or prawn)", "desc": "Pizza, seafood", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 320, "name": "Pizza, cheese stuffed crust", "desc": "Pizza, cheese stuffed crust (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 322, "name": "Murtabak ayam", "desc": "Malaysian murtabak: thick square folded pan-fried bread stuffed with egg, onion and minced meat, with curry and pickled onion, with ayam (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 323, "name": "Roti john", "desc": "Malaysian roti john: a long bread roll fried with egg and minced meat, topped with chilli sauce and mayonnaise (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 326, "name": "Esquire Kitchen salted fish fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with salted fish (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 328, "name": "Wok Hey black pepper beef fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with black pepper beef (1 box)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 329, "name": "Nando's spicy rice", "desc": "spicy rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 330, "name": "DIN pork chop fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with pork chop (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 331, "name": "Prawn and egg white fried cauliflower rice, light", "desc": "Chinese Prawn and egg white fried cauliflower rice, light (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 336, "name": "Long Jing tofu with minced pork", "desc": "Chinese tofu with minced pork (1 plate with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 338, "name": "Peng Chu Hunan chili pork", "desc": "Chinese Hunan chili pork (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 339, "name": "Nong Geng Ji pepper fried pork", "desc": "Chinese pepper fried pork (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 340, "name": "Tofu and minced chicken with long beans", "desc": "Chinese Tofu and minced chicken with long beans", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 341, "name": "Fish slice soup with tofu and tomato", "desc": "Chinese Fish slice soup with tofu and tomato (1 bowl, no rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 346, "name": "Mixed vegetable minestrone", "desc": "Mixed vegetable minestrone", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 347, "name": "Minced pork soup", "desc": "Chinese Minced pork soup (1 bowl, with gourd or greens)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 348, "name": "Fried fish cake", "desc": "Chinese Fried fish cake (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 350, "name": "Chicken quesadilla", "desc": "Chicken quesadilla (1 tortilla, 4 wedges)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 352, "name": "Shojikiya beef sukiyaki set", "desc": "Japanese beef sukiyaki set", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 355, "name": "Samgyeopsal set with rice", "desc": "Korean Samgyeopsal set with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 356, "name": "Bulgogi with rice", "desc": "Korean Bulgogi with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 357, "name": "Kimchi jjigae with rice", "desc": "Korean Kimchi jjigae with rice (1 pot, 1 bowl rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 359, "name": "Kimchi tofu stew, light", "desc": "Korean Kimchi tofu stew, light (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 360, "name": "Sundubu jjigae with rice", "desc": "Korean Sundubu jjigae with rice (1 pot, 1 bowl rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 362, "name": "MyeongDong topokki", "desc": "Korean topokki (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 363, "name": "Mr Dakgalbi cheese dakgalbi, per person", "desc": "Korean cheese dakgalbi, per person (With rice cakes and cabbage)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 365, "name": "Ramyeon with egg", "desc": "Korean Ramyeon with egg (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 366, "name": "Jajangmyeon", "desc": "Korean jajangmyeon: thick noodles covered in glossy black bean sauce with diced pork and onion, with cucumber strips (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 367, "name": "Samgyetang", "desc": "Korean Samgyetang (half chicken with rice in soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 369, "name": "K Fry Yangnyeom chicken", "desc": "Korean Yangnyeom chicken (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 371, "name": "Seafood pajeon", "desc": "Korean Seafood pajeon (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 372, "name": "Army stew (budae jjigae)", "desc": "Korean Army stew (1 person's share with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 373, "name": "Mookata (Thai BBQ), typical meal", "desc": "Thai Mookata, typical meal (1 person)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 374, "name": "Yakiniku, typical meal", "desc": "Japanese Yakiniku, typical meal (1 person with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 377, "name": "Pizza Hut chicken wings", "desc": "Pizza Hut chicken wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 378, "name": "BBQ pork ribs", "desc": "BBQ pork ribs (half rack)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 379, "name": "Lamb skewers (BBQ)", "desc": "Chinese Lamb skewers (5 skewers)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 380, "name": "Bar B Q Plaza pork set, per person", "desc": "Thai pork set, per person (Grilled pork slices with vegetables)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 381, "name": "Rocku yakiniku beef set, per person", "desc": "Japanese beef set, per person (Grilled beef, rice and sides)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 383, "name": "Grilled chicken wings (BBQ)", "desc": "Malaysian Grilled chicken wings (2 wings)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 385, "name": "Otak-otak", "desc": "otak-otak: orange spiced fish paste grilled in banana leaf parcels (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 386, "name": "Ikan pari bakar with rice", "desc": "Malaysian pari bakar with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 387, "name": "Lok lok", "desc": "lok lok skewers of fish balls, sausages and tofu with satay and chilli sauce (10 sticks with sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 389, "name": "Double cheeseburger", "desc": "Double cheeseburger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 390, "name": "Double beef burger", "desc": "Double beef burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 391, "name": "Mushroom Swiss burger", "desc": "Mushroom Swiss burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 392, "name": "Spicy Chicken McDeluxe", "desc": "Spicy Chicken McDeluxe (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 393, "name": "Chicken burger, crispy", "desc": "Chicken burger, crispy (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 394, "name": "Fish burger", "desc": "Fish burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 395, "name": "Ramly burger, ayam special", "desc": "Malaysian Ramly burger, ayam special (1 burger with egg wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 400, "name": "Burger King onion rings", "desc": "Burger King onion rings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 401, "name": "Burger King fries", "desc": "Burger King fries", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 402, "name": "Nando's peri-peri chips", "desc": "peri-peri chips", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 403, "name": "Texas Chicken cajun fries", "desc": "Chicken cajun fries", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 404, "name": "Carl's Jr onion rings", "desc": "onion rings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 405, "name": "Happy Potato cheese potato balls", "desc": "Korean cheese potato balls (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 406, "name": "Potato Corner flavoured fries, regular", "desc": "flavoured fries, regular (Cheese or BBQ)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 407, "name": "Chica Bonita nachos with cheese", "desc": "nachos with cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 408, "name": "Hotcakes with syrup", "desc": "Hotcakes with syrup (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 409, "name": "Apple pie (McD)", "desc": "Apple pie (1 pie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 412, "name": "The Barn fried chicken", "desc": "fried chicken (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 414, "name": "KFC Hot & Spicy, breast", "desc": "Hot & Spicy, breast (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 415, "name": "Cheezy wedges, large", "desc": "Cheezy wedges, large", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 416, "name": "KFC nuggets", "desc": "nuggets (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 418, "name": "Burger King nuggets", "desc": "Burger King nuggets (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 419, "name": "Whipped potato with gravy", "desc": "Whipped potato with gravy", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 420, "name": "Texas Chicken mashed potato", "desc": "Chicken mashed potato", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 421, "name": "Coleslaw", "desc": "Coleslaw", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 422, "name": "Texas Chicken, 2 pieces", "desc": "Chicken, 2 pieces (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 423, "name": "Honey butter biscuit", "desc": "Honey butter biscuit (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 425, "name": "Root beer float", "desc": "Root beer float (1 mug)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 428, "name": "Wang Donkas pork cutlet", "desc": "Korean pork cutlet (With rice and sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 429, "name": "Wang Donkas cheese donkas", "desc": "Korean cheese donkas (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 430, "name": "Salmon don", "desc": "Japanese Salmon don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 431, "name": "Unagi don", "desc": "Japanese Unagi don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 432, "name": "Tempura udon", "desc": "Japanese Tempura udon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 433, "name": "Chicken karaage", "desc": "Japanese Chicken karaage (5 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 434, "name": "Okonomiyaki", "desc": "Japanese Okonomiyaki (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 435, "name": "Yakitori", "desc": "Japanese Yakitori (3 skewers)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 436, "name": "California roll", "desc": "Japanese California roll (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 437, "name": "Salmon maki", "desc": "Japanese Salmon maki (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 438, "name": "Kappa maki (cucumber)", "desc": "Japanese Kappa maki (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 439, "name": "Spicy salmon roll", "desc": "Japanese Spicy salmon roll (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 440, "name": "Fried ebi roll", "desc": "Japanese Fried ebi roll (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 441, "name": "Cold soba", "desc": "Japanese Cold soba (1 tray)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 444, "name": "Black Canyon green curry with rice", "desc": "Thai green curry with chicken and eggplant in a bowl beside rice, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 446, "name": "Chakri massaman chicken curry", "desc": "Malaysian chicken curry with potatoes in orange gravy; Thai massaman curry with potato and chicken beside rice (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 447, "name": "Amarin green curry chicken", "desc": "Malaysian chicken curry with potatoes in orange gravy, with green (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 448, "name": "Celadon green curry chicken", "desc": "Malaysian chicken curry with potatoes in orange gravy, with green (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 449, "name": "Chicken curry with potatoes and rice", "desc": "Malaysian chicken curry with potatoes in orange gravy, with potatoes and rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 450, "name": "Som tam", "desc": "Thai Som tam (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 452, "name": "Dragon-i la mian in spicy soup", "desc": "Chinese hand-pulled noodles, with in spicy soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 454, "name": "NakNak ramyeon with egg", "desc": "Korean ramyeon with egg (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 456, "name": "Yunnan spicy rice noodles", "desc": "Chinese Yunnan spicy rice noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 457, "name": "Spicy bone soup rice noodles", "desc": "Chinese Spicy bone soup rice noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 458, "name": "Thai fried rice with chicken", "desc": "golden egg fried rice with spring onion and diced vegetables, with thai with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 460, "name": "ChaTraMue Thai tea", "desc": "ChaTraMue Thai tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 461, "name": "Moo ping with sticky rice", "desc": "Thai Moo ping with sticky rice (3 skewers, 1 pack rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 462, "name": "Chicken chop with black pepper sauce", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw, with black pepper sauce (1 plate with fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 463, "name": "Lamb chop with fries", "desc": "Lamb chop with fries (2 chops)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 464, "name": "ANTE kurobuta pork chop", "desc": "kurobuta pork chop (1 plate with sides)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 465, "name": "Dave's lamb shank", "desc": "lamb shank (With mash)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 467, "name": "Fish & Co clam chowder", "desc": "clam chowder (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 468, "name": "Sushi Zanmai chawanmushi", "desc": "Japanese Sushi Zanmai chawanmushi (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 469, "name": "Pizza Hut garlic bread", "desc": "Pizza Hut garlic bread (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 470, "name": "Pizza Hut mushroom soup", "desc": "Pizza Hut mushroom soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 471, "name": "Big breakfast", "desc": "Big breakfast (eggs, sausage, toast, beans, hash brown)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 472, "name": "Avocado toast with egg", "desc": "Avocado toast with egg (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 473, "name": "Beef burger with fries", "desc": "Beef burger with fries (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 474, "name": "Seafood marinara pasta", "desc": "Seafood marinara pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 475, "name": "Creamy mushroom chicken pasta", "desc": "Creamy mushroom chicken pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 479, "name": "Stuff'd burrito", "desc": "burrito (1 burrito)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 480, "name": "Tim Hortons Farmer's breakfast wrap", "desc": "breakfast wrap (1 wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 481, "name": "La Boca beef tacos", "desc": "beef tacos (3 tacos)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 482, "name": "Chica Bonita beef burrito", "desc": "beef burrito (1 burrito)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 483, "name": "Smoothie bowl", "desc": "Smoothie bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 485, "name": "Bee Cheng Hiang bak kwa, pork", "desc": "Chinese bak kwa, pork", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 486, "name": "Siew yuk rice", "desc": "slices of crispy-skinned roast pork belly, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 487, "name": "Sweet and sour pork with rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with pork with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 488, "name": "Putien sweet and sour pork with lychee", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with pork with lychee", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 489, "name": "Sweet and sour chicken with rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with chicken with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 490, "name": "Kung pao chicken with rice", "desc": "diced chicken stir-fried with dried red chillies and peanuts, with chicken with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 491, "name": "Salted egg chicken rice", "desc": "fried chicken pieces coated in golden salted egg yolk sauce with curry leaves, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 492, "name": "Butter prawns", "desc": "Chinese Butter prawns (6 prawns)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 493, "name": "Bar B Q Plaza seafood set, per person", "desc": "Thai seafood set, per person (Prawns, squid and fish)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 494, "name": "Tangsuyuk (sweet and sour pork)", "desc": "Korean Tangsuyuk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 495, "name": "Andaman grilled seafood platter, per person", "desc": "Thai grilled seafood platter, per person (Prawns, squid, fish and rice)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 496, "name": "Fei Fei Crab spicy crab pot, per person", "desc": "Chinese spicy crab pot, per person (Crab, prawns, corn and potato)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 498, "name": "Butter garlic prawns with rice", "desc": "Butter garlic prawns with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 499, "name": "Loh mee", "desc": "loh mee: yellow noodles in thick dark starchy gravy with braised pork and egg (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 500, "name": "Sarawak laksa", "desc": "Sarawak laksa: rice vermicelli in light orange sambal soup with prawns, shredded chicken and omelette strips (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 501, "name": "Kolo mee", "desc": "Sarawak kolo mee: springy curly egg noodles with slices of red char siu and minced pork (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 502, "name": "Chilli pan mee", "desc": "dry pan mee noodles with minced pork, crispy fried anchovies, a poached egg and dark chilli flakes on the side (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 503, "name": "Claypot yee mee", "desc": "claypot yee mee: crispy egg noodles in a claypot of soup with egg, prawns and greens (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 504, "name": "Economy bee hoon", "desc": "thin rice vermicelli, with economy (1 plate with egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 508, "name": "Cili Kampung nasi ayam bakar", "desc": "grilled marinated chicken with charred edges, with nasi (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 510, "name": "Longjing Dongpo pork rice", "desc": "a glossy cube of dark braised pork belly, with rice (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 511, "name": "Nasi kandar with fried chicken and curry", "desc": "Penang nasi kandar: white rice flooded with mixed dark curry gravies, fried chicken, okra and a boiled egg, with fried chicken and curry (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 512, "name": "Hameed nasi kandar", "desc": "Penang nasi kandar: white rice flooded with mixed dark curry gravies, fried chicken, okra and a boiled egg (1 plate with fried chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 513, "name": "Masala King mutton biryani", "desc": "long-grain yellow and orange spiced biryani rice with fried shallots, with mutton (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 514, "name": "Ayam percik with rice", "desc": "grilled ayam percik chicken glazed with orange spiced coconut sauce, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 515, "name": "Nasi goreng pattaya", "desc": "fried rice wrapped in a thin yellow omelette envelope, drizzled with chilli sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 516, "name": "Nasi goreng USA", "desc": "fried rice topped with sweet and sour sauce, fried chicken pieces and a fried egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 517, "name": "Sambal sotong with rice", "desc": "squid rings in glossy red sambal, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 519, "name": "Cheese naan", "desc": "a soft blistered naan flatbread, with cheese (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 521, "name": "Butter chicken with naan", "desc": "creamy orange butter chicken curry with a piece of naan; a soft blistered naan flatbread (1 bowl, 1 naan)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 523, "name": "Vadai", "desc": "two golden doughnut-shaped fried lentil vadai (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 524, "name": "Puri with potato masala", "desc": "puffed golden fried puri bread with potato masala, with potato masala (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 527, "name": "Ramen, miso", "desc": "Japanese Ramen, miso (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 528, "name": "Ramen, shio", "desc": "Japanese Ramen, shio (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 529, "name": "Ramen, spicy miso", "desc": "Japanese Ramen, spicy miso (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 530, "name": "Ramen, tantanmen", "desc": "Japanese Ramen, tantanmen (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 531, "name": "Ramen, chicken paitan", "desc": "Japanese Ramen, chicken paitan (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 532, "name": "Tsukemen (dipping ramen)", "desc": "Japanese Tsukemen (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 533, "name": "Extra chashu", "desc": "Japanese Extra chashu (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 534, "name": "Ajitama (ramen egg)", "desc": "Japanese Ajitama (1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 536, "name": "Shin Ramyun", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 537, "name": "Shin Ramyun with egg and cheese", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 538, "name": "Buldak hot chicken ramen", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 539, "name": "Buldak carbonara ramen", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 540, "name": "Jjapaguri (ram-don)", "desc": "Korean Jjapaguri (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 541, "name": "Kimchi ramyeon", "desc": "Korean Kimchi ramyeon (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 542, "name": "Seafood ramyeon (jjamppong style)", "desc": "Korean Seafood ramyeon (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 543, "name": "Cheese ramyeon", "desc": "Korean Cheese ramyeon (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 544, "name": "Instant noodles, curry (Maggi)", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 545, "name": "Indomie mi goreng", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 546, "name": "Cup noodles", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 547, "name": "Spaghetti aglio olio", "desc": "Spaghetti aglio olio (1 plate, no meat)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 548, "name": "Spaghetti aglio olio with prawns", "desc": "Spaghetti aglio olio with prawns (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 549, "name": "Spaghetti pomodoro", "desc": "Spaghetti pomodoro (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 550, "name": "Spaghetti arrabbiata", "desc": "Spaghetti arrabbiata (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 551, "name": "Spaghetti pesto with chicken", "desc": "Spaghetti pesto with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 552, "name": "Spaghetti alfredo", "desc": "Spaghetti alfredo (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 553, "name": "Spaghetti vongole", "desc": "Spaghetti vongole (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 554, "name": "Spaghetti meatballs", "desc": "Spaghetti meatballs (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 555, "name": "Truffle mushroom pasta", "desc": "Truffle mushroom pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 556, "name": "Salted egg pasta", "desc": "Salted egg pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 557, "name": "Tom yum pasta", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 558, "name": "Penne arrabbiata with chicken", "desc": "Penne arrabbiata with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 559, "name": "Baked cheese pasta", "desc": "Baked cheese pasta (1 dish)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 560, "name": "Mac and cheese with chicken", "desc": "Mac and cheese with chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 561, "name": "Fettuccine carbonara with mushroom", "desc": "Fettuccine carbonara with mushroom (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 562, "name": "Ravioli in tomato sauce", "desc": "Ravioli in tomato sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 563, "name": "Nasi goreng cina", "desc": "light golden Chinese-style egg fried rice with peas, carrot and spring onion (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 564, "name": "Nasi goreng ikan masin", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with ikan masin (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 565, "name": "Nasi goreng tom yum", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables; Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 566, "name": "Nasi goreng ayam", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with ayam (1 plate with fried chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 567, "name": "Nasi goreng seafood", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with seafood (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 568, "name": "Nasi goreng belacan", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with belacan (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 569, "name": "Nasi goreng daging merah", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with daging merah (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 570, "name": "Yang chow fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with yang chow (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 571, "name": "Garlic fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with garlic (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 574, "name": "Little Cravings nasi lemak with fried chicken", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with fried chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 575, "name": "Se'Indonesia nasi ayam penyet", "desc": "Indonesian ayam penyet: smashed fried chicken with red sambal, rice, tofu, tempeh and cucumber, with nasi (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 577, "name": "Jollibee Chickenjoy, 1 piece with rice", "desc": "Chickenjoy, 1 piece with rice (1 piece, rice, gravy)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 578, "name": "Dirty duck with rice", "desc": "Malaysian Dirty duck with rice (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 579, "name": "Meriah platter", "desc": "Malaysian platter (Pandan rice, fried chicken, beef rendang)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 580, "name": "Nasi lemak sambal sotong", "desc": "squid rings in glossy red sambal; Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 581, "name": "Nasi lemak telur mata", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with telur mata (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 582, "name": "Nasi lemak kerang", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with kerang (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 583, "name": "Roti boom Milo", "desc": "a glass of chocolate malt drink, with roti boom (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 585, "name": "Veggie burger", "desc": "Veggie burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 586, "name": "Tuna nigiri", "desc": "Japanese Tuna nigiri (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 587, "name": "Tamago nigiri", "desc": "Japanese Tamago nigiri (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 588, "name": "Unagi nigiri", "desc": "Japanese Unagi nigiri (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 589, "name": "Inari sushi", "desc": "Japanese Inari sushi (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 590, "name": "Makii Makii onigiri, tuna mayo", "desc": "Japanese onigiri, tuna mayo (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 591, "name": "Salmon mentai aburi", "desc": "Japanese Salmon mentai aburi (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 593, "name": "Shucked fresh oysters", "desc": "fresh oysters (6 oysters)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 594, "name": "Chirashi don", "desc": "Japanese Chirashi don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 595, "name": "Cappuccino", "desc": "Cappuccino", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 596, "name": "Flat white", "desc": "Flat white", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 599, "name": "Antipodean flat white", "desc": "flat white", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 600, "name": "illy cappuccino", "desc": "cappuccino", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 601, "name": "Latte with oat milk, iced", "desc": "Latte with oat milk, iced", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 603, "name": "Black Canyon iced cappuccino", "desc": "iced cappuccino (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 606, "name": "Gula melaka latte, iced", "desc": "Gula melaka latte, iced", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 607, "name": "Americano, iced", "desc": "Americano, iced", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 615, "name": "Milo ais kurang manis", "desc": "a glass of chocolate malt drink, with ais kurang manis (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 616, "name": "Milo dinosaur", "desc": "a tall glass of iced Milo topped with a mound of Milo powder (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 618, "name": "Thai milk tea, iced", "desc": "a plastic cup of milk tea, with thai , iced (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 619, "name": "Mr Tuk Tuk Thai iced tea", "desc": "Thai iced tea (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 622, "name": "Chicha San Chen Dong Ding oolong milk tea", "desc": "a plastic cup of milk tea, with dong ding oolong", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 623, "name": "Matcha milk tea with pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with matcha", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 625, "name": "ChaTraMue Thai green tea", "desc": "ChaTraMue Thai green tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 626, "name": "Chocolate milk tea with pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 627, "name": "Fruit tea with pearls", "desc": "with black tapioca pearls at the bottom and a thick straw, with fruit tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 628, "name": "Cheese foam tea", "desc": "Cheese foam tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 629, "name": "Tea macchiato", "desc": "Tea macchiato", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 634, "name": "Banh mi", "desc": "Banh mi (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 635, "name": "Vietnamese fresh spring rolls", "desc": "golden crispy fried spring rolls, with fresh s (2 rolls with dip)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 636, "name": "Beancurd skin roll, steamed", "desc": "Chinese Beancurd skin roll, steamed (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 637, "name": "Udon, kitsune", "desc": "Japanese Udon, kitsune (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 638, "name": "Udon, beef (niku)", "desc": "Japanese Udon, beef (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 639, "name": "Udon, curry", "desc": "Japanese Udon, curry (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 640, "name": "Yaki udon", "desc": "Japanese Yaki udon (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 641, "name": "Yakisoba", "desc": "Japanese Yakisoba (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 642, "name": "Fish curry with rice", "desc": "fish curry with okra in red gravy, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 643, "name": "Mutton curry with rice", "desc": "dark mutton curry, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 644, "name": "Vegetable curry with rice", "desc": "Indian Vegetable curry with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 645, "name": "Japanese curry rice, chicken", "desc": "Japanese curry rice, chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 646, "name": "Japanese curry rice, beef", "desc": "Japanese curry rice, beef (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 647, "name": "Massaman curry with rice", "desc": "Thai massaman curry with potato and chicken beside rice, with curry with rice (1 bowl curry, 1 cup rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 648, "name": "Red curry chicken with rice", "desc": "Malaysian chicken curry with potatoes in orange gravy, with red with rice (1 bowl curry, 1 cup rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 649, "name": "Kari kepala ikan", "desc": "fish head curry with okra and eggplant in a clay pot (1 person's share with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 650, "name": "Fish porridge", "desc": "smooth Cantonese rice porridge, with fish (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 651, "name": "Pork porridge", "desc": "smooth Cantonese rice porridge, with pork (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 652, "name": "Plain porridge with side dishes (Teochew)", "desc": "smooth Cantonese rice porridge, with plain with side dishes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 653, "name": "Bubur lambuk", "desc": "rice porridge, with lambuk (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 654, "name": "Oat porridge, savoury", "desc": "smooth Cantonese rice porridge, with oat , savoury (1 bowl with egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 655, "name": "Scrambled eggs", "desc": "Scrambled eggs (2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 656, "name": "Fried egg", "desc": "Fried egg (1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 658, "name": "Poached eggs", "desc": "Poached eggs (2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 659, "name": "Half-boiled eggs", "desc": "a small bowl of two runny soft-boiled eggs with soy sauce and white pepper (2 eggs with soy sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 660, "name": "Telur bungkus", "desc": "an omelette parcel with sauce (1 piece with sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 661, "name": "Tuna sandwich", "desc": "Tuna sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 662, "name": "Chicken sandwich, grilled", "desc": "Chicken sandwich, grilled (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 663, "name": "Ham and cheese sandwich", "desc": "Ham and cheese sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 664, "name": "Grilled cheese sandwich", "desc": "Grilled cheese sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 665, "name": "BLT sandwich", "desc": "BLT sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 666, "name": "Caesar salad, no chicken", "desc": "Caesar salad, no chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 667, "name": "Greek salad", "desc": "Greek salad (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 668, "name": "Grilled chicken salad", "desc": "Grilled chicken salad (1 bowl, dressing on the side)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 669, "name": "Tuna salad", "desc": "Tuna salad (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 670, "name": "Poke bowl, salmon", "desc": "Japanese Poke bowl, salmon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 671, "name": "Ayam goreng berempah", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden, with berempah (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 672, "name": "Ayam goreng kunyit", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden, with kunyit (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 673, "name": "Chicken tenders", "desc": "Chicken tenders (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 674, "name": "Chicken wings, fried", "desc": "Chicken wings, fried (3 wings)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 675, "name": "Roast chicken, quarter", "desc": "Roast chicken, quarter (1 leg quarter with skin)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 676, "name": "Nasi minyak", "desc": "golden ghee rice with raisins and cashew nuts (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 677, "name": "Nasi tomato", "desc": "orange tomato rice with green peas (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 678, "name": "Nasi briyani, rice only", "desc": "Malaysian nasi briyani: long-grain yellow and orange spiced rice with a chicken curry, papadam and acar, with rice only (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 680, "name": "Japanese rice", "desc": "Japanese rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 681, "name": "Cauliflower rice", "desc": "Cauliflower rice (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 682, "name": "Butter toast", "desc": "Butter toast (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 683, "name": "Peanut butter toast", "desc": "Peanut butter toast (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 684, "name": "Cottage cheese toast with tomato", "desc": "Cottage cheese toast with tomato (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 688, "name": "Tim Tam Dark Chocolate", "desc": "Tim Tam Dark Chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 689, "name": "KitKat Matcha", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 690, "name": "KitKat Strawberry", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 693, "name": "Cadbury Dairy Milk Oreo", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 697, "name": "Beryl's chocolate", "desc": "chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 698, "name": "Pocky Chocolate", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 699, "name": "Choki Choki", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 701, "name": "Pocky Matcha", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 702, "name": "Pocky Cookies & Cream", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 703, "name": "Pocky Banana", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 704, "name": "Pocky Almond Crush", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 705, "name": "Yan Yan Chocolate", "desc": "small biscuits with chocolate dip", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 707, "name": "Oreo Strawberry Creme", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 708, "name": "Oreo Chocolate Creme", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 709, "name": "Chipsmore Original", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 710, "name": "Chipsmore Double Chocolate", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 712, "name": "Munchy's Oat Krunch Strawberry & Blackcurrant", "desc": "oat cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 713, "name": "Subway chocolate chip cookie", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 715, "name": "Tokyo Milk Cheese Factory cheese cookies", "desc": "Japanese cheese cookies (2 cookies)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 717, "name": "Cafe Kitsune fox shortbread", "desc": "Cafe Kitsune fox shortbread (1 cookie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 718, "name": "Ferrero Rocher", "desc": "round chocolate truffles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 719, "name": "Le Bourne chocolate pralines", "desc": "pralines (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 721, "name": "Kinder Chocolate", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 722, "name": "Toblerone", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 723, "name": "Snickers", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 724, "name": "M&M's Milk Chocolate", "desc": "colourful sugar-coated chocolate buttons", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 727, "name": "Super Ring Cheese", "desc": "Super Ring Cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 728, "name": "Hershey's Kisses", "desc": "round chocolate truffles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 729, "name": "Lindt Lindor", "desc": "round chocolate truffles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 730, "name": "Hello Panda Chocolate", "desc": "small biscuits with chocolate dip", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 731, "name": "Twisties Chicken", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 732, "name": "Potato chips", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 734, "name": "Koala's March Chocolate", "desc": "March Chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 735, "name": "Mister Potato Hot & Spicy", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 737, "name": "Doritos Nacho Cheese", "desc": "Doritos Nacho Cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 738, "name": "Mamee Monster Hot & Spicy", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 739, "name": "Popo Fish Muruku", "desc": "Popo Fish Muruku", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 740, "name": "Lotte Choco Pie", "desc": "a chocolate-coated marshmallow cake", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 741, "name": "Julie's Peanut Butter Sandwich", "desc": "Peanut Butter Sandwich", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 742, "name": "Julie's Cheese Sandwich", "desc": "Cheese Sandwich", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 744, "name": "Lexus Chocolate Cream Sandwich", "desc": "Lexus Chocolate Cream Sandwich", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 745, "name": "Wholemeal crackers with cheese", "desc": "Wholemeal crackers with cheese (4 crackers, 1 slice)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 746, "name": "Julie's Love Letters", "desc": "Love Letters", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 747, "name": "Apollo Chocolate Layer Cake", "desc": "small layer cake squares", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 748, "name": "Apollo Pandan Layer Cake", "desc": "small layer cake squares", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 749, "name": "Twisties Cheese", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 750, "name": "Mamee Snek Ku", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 752, "name": "Mister Potato Original", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 753, "name": "Mister Potato Barbecue", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 754, "name": "Mamee Monster Barbecue", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 755, "name": "Twisties Barbecue Curry", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 756, "name": "Pringles Original", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 757, "name": "Pringles Sour Cream & Onion", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 758, "name": "Pringles Barbecue", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 759, "name": "Pringles Cheesy Cheese", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 760, "name": "Jacker potato chips", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 761, "name": "Pringles Hot & Spicy", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 762, "name": "Lay's Sour Cream & Onion", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 763, "name": "Lay's Nori Seaweed", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 764, "name": "Calbee Jagabee", "desc": "Calbee Jagabee", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 765, "name": "Ding Dang", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 766, "name": "Mentos", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 767, "name": "Hacks", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 769, "name": "Haw flakes", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 770, "name": "Absolute Thai mango sticky rice", "desc": "Thai mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 772, "name": "Jatujak mango sticky rice", "desc": "Thai mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 773, "name": "Yomie's purple rice yogurt", "desc": "purple rice yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 774, "name": "Pai tee", "desc": "pai tee: small crispy golden top-hat cups filled with shredded vegetables and a prawn (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 776, "name": "Snowflake taro balls dessert", "desc": "Chinese taro balls dessert (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 777, "name": "bhc cheese balls", "desc": "Korean cheese balls (5 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 778, "name": "Pineapple tarts", "desc": "small golden pineapple tarts with jam (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 781, "name": "Zok bamboo noodles with char siu", "desc": "slices of red-edged glossy char siu barbecued pork, with bamboo noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 785, "name": "Mak's Chee wonton noodle soup", "desc": "thin springy egg noodles with slices of char siu and small boiled wontons, with soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 786, "name": "Ki Ki dry noodles with scallion oil", "desc": "Chinese dry noodles with scallion oil (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 787, "name": "Soong Kee beef ball noodles, dry", "desc": "Chinese ball noodles, dry (1 bowl with soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 789, "name": "Jipinhe scone, plain", "desc": "scone, plain (1 scone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 790, "name": "Jipinhe chocolate scone", "desc": "chocolate scone (1 scone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 791, "name": "Petite Maison pain au chocolat", "desc": "pain au chocolat (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 794, "name": "Elementary beef pie", "desc": "beef pie (1 pie with mash)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 799, "name": "Baskin Robbins ice cream cake", "desc": "ice cream cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 800, "name": "Chili's Southwestern eggrolls", "desc": "Southwestern eggrolls (1 starter)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 806, "name": "Gordon Ramsay beef Wellington", "desc": "beef Wellington (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 807, "name": "Weissbrau pork knuckle", "desc": "pork knuckle (Half portion with sauerkraut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 808, "name": "Corks Out steak with fries", "desc": "steak with fries (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 809, "name": "Grilled lean beef with asparagus and mash", "desc": "Grilled lean beef with asparagus and mash (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 810, "name": "De Irrfan's roti john", "desc": "Malaysian roti john: a long bread roll fried with egg and minced meat, topped with chilli sauce and mayonnaise (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 812, "name": "Din Tai Fung chicken xiao long bao", "desc": "xiao long bao soup dumplings with fine pleats in a bamboo steamer, with chicken (5 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 813, "name": "Dragon-i deep-fried dumplings", "desc": "Chinese pleated dumplings, with deep-fried (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 816, "name": "B.O.D steamed bun with meat", "desc": "Chinese steamed bun with meat (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 819, "name": "Mr Bean soya pancake with peanut", "desc": "Chinese soya pancake with peanut (1 pancake)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 820, "name": "dipndip chocolate crepe", "desc": "chocolate crepe (1 crepe)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 823, "name": "Beard Papa chocolate cream puff", "desc": "Japanese chocolate cream puff (1 puff)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 826, "name": "Pan-fried pork dumplings", "desc": "Chinese pleated dumplings, with pan-fried pork (8 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 827, "name": "K Fry Berry-Matcha Mochi Bingsoo", "desc": "Korean Berry-Matcha Mochi Bingsoo (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 834, "name": "Ma Yi Lan Lanzhou beef noodles", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy, with lanzhou s (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 835, "name": "Nippori matcha soft serve", "desc": "Japanese matcha soft serve (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 838, "name": "Boat Noodle mango sticky rice", "desc": "Thai Noodle mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 839, "name": "ParaThai mango sticky rice", "desc": "Thai mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 840, "name": "Kuih, 2 pieces", "desc": "colourful Malaysian kuih cakes, such as layered kuih lapis and green onde-onde, with 2 pieces (Onde-onde, kuih lapis or seri muka)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 843, "name": "Yin's sourdough loaf slice", "desc": "loaf slice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 844, "name": "Shae.co bagel sandwich", "desc": "bagel sandwich (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 846, "name": "Pho Vietz fried spring rolls", "desc": "golden crispy fried spring rolls, with fried s (3 rolls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 848, "name": "Black Tap CrazyShake, cookies and cream", "desc": "CrazyShake, cookies and cream (1 shake)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 852, "name": "Dunkin' chocolate frosted donut", "desc": "chocolate frosted donut (1 donut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 855, "name": "Pork and chive dumplings, boiled", "desc": "Chinese pleated dumplings, with pork and chive , boiled (10 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 856, "name": "Vegetable dumplings, steamed", "desc": "Chinese pleated dumplings, with vegetable , steamed (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa, food, plate", "n": 1}, {"id": 858, "name": "Burger King sundae, chocolate", "desc": "Burger King sundae, chocolate (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 859, "name": "108 Matcha Saro matcha soft serve", "desc": "Japanese matcha soft serve (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 861, "name": "Gordon Ramsay sticky toffee pudding", "desc": "sticky toffee pudding (1 dessert)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 863, "name": "Mango pudding", "desc": "Chinese Mango pudding (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 864, "name": "Din Tai Fung spicy wontons", "desc": "small boiled wontons with silky wrappers, with spicy s (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 867, "name": "Longan and sea coconut", "desc": "Chinese Longan and sea coconut (1 bowl, iced)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 869, "name": "dipndip waffle with chocolate", "desc": "waffle with chocolate (1 waffle)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 873, "name": "Mille crepe, original", "desc": "Mille crepe, original (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 874, "name": "Jollibee peach mango pie", "desc": "peach mango pie (1 pie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 875, "name": "Apple pie", "desc": "Apple pie (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 876, "name": "Chicken pie", "desc": "Chicken pie (1 pie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 877, "name": "Quiche", "desc": "Quiche (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 881, "name": "Gindaco takoyaki, cheese mentai", "desc": "Japanese takoyaki, cheese mentai (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 882, "name": "Okonomi by Tokyo Don okonomiyaki", "desc": "Japanese okonomiyaki (1 pancake)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 883, "name": "Putien Bian Rou soup (meat dumplings)", "desc": "Chinese Bian Rou soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 884, "name": "Yuen Kee pork and chive dumplings, boiled", "desc": "Chinese pleated dumplings, with pork and chive , boiled (10 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 887, "name": "Sambal kangkung", "desc": "Malaysian Sambal kangkung", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 888, "name": "Stir-fried greens", "desc": "Stir-fried greens (1 plate (kangkung, sawi))", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 890, "name": "Baked salmon with sweet potato and greens", "desc": "Baked salmon with sweet potato and greens", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 892, "name": "Dumplings in soup with greens", "desc": "Chinese pleated dumplings, with in soup with greens", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 893, "name": "Konjac noodle soup with prawns and greens", "desc": "Chinese Konjac noodle soup with prawns and greens (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 894, "name": "Soba with chicken and edamame", "desc": "Japanese Soba with chicken and edamame (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 896, "name": "Lotus root soup, vegetarian", "desc": "Chinese Lotus root soup, vegetarian", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 898, "name": "Steamed egg with minced pork and rice", "desc": "Chinese Steamed egg with minced pork and rice", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 899, "name": "Steamed egg with minced chicken, rice", "desc": "Chinese Steamed egg with minced chicken, rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 901, "name": "Chicken breast salad wrap", "desc": "Chicken breast salad wrap", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 902, "name": "Vietnamese prawn rice paper rolls", "desc": "Vietnamese prawn rice paper rolls (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 903, "name": "Tuna and egg salad wrap", "desc": "Tuna and egg salad wrap (1 wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 904, "name": "Miso salmon and tofu soup with rice", "desc": "Japanese Miso salmon and tofu soup with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 905, "name": "Spinach and egg drop soup with rice", "desc": "Chinese Spinach and egg drop soup with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 906, "name": "Miso soup with tofu, rice and egg", "desc": "Japanese Miso soup with tofu, rice and egg (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 907, "name": "Egg white omelette with spinach and mushrooms", "desc": "Egg white omelette with spinach and mushrooms (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 908, "name": "Chicken and vegetable pasta", "desc": "Chicken and vegetable pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 909, "name": "Baked chicken thigh with potatoes and salad", "desc": "Baked chicken thigh with potatoes and salad (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 910, "name": "Sardine sandwich", "desc": "Malaysian Sardine sandwich (2 slices wholemeal)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 911, "name": "Banana", "desc": "Banana", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 912, "name": "Papaya", "desc": "Papaya (1 cup cubes)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 913, "name": "Apple", "desc": "Apple", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 914, "name": "Guava", "desc": "Guava (1 cup slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 915, "name": "Watermelon", "desc": "Watermelon (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 916, "name": "Orange", "desc": "Orange", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 917, "name": "Mixed cut fruit", "desc": "Mixed cut fruit (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 918, "name": "Boiled egg", "desc": "Boiled egg (1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 919, "name": "Boiled eggs", "desc": "Boiled eggs (2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 920, "name": "Blanched greens with oyster sauce", "desc": "Blanched greens with oyster sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 921, "name": "Ulam with sambal on the side", "desc": "fresh Malay herbs and raw vegetables with sambal belacan, with sambal on the side (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 922, "name": "Garden salad", "desc": "Garden salad (1 bowl, light dressing)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 923, "name": "Corn cup", "desc": "Corn cup (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 924, "name": "Edamame", "desc": "Japanese Edamame", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 925, "name": "Chawanmushi", "desc": "Japanese Chawanmushi (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 926, "name": "Miso soup", "desc": "Japanese Miso soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 927, "name": "Unsweetened soy milk", "desc": "a glass of soy milk, with unsweetened", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 928, "name": "Plain yogurt", "desc": "Plain yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 929, "name": "Greek yogurt", "desc": "Greek yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 930, "name": "Low-fat milk", "desc": "Low-fat milk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 931, "name": "Mixed nuts", "desc": "Mixed nuts", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 932, "name": "Tau fu fah, less syrup", "desc": "Chinese less syrup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 933, "name": "Steamed corn", "desc": "Steamed corn (1 cob)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 934, "name": "Popiah, 1 roll", "desc": "fresh popiah: soft white crepe rolls filled with braised turnip, egg and lettuce, cut in pieces, with 1 roll (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 936, "name": "Apam balik", "desc": "apam balik: thick folded peanut pancake with crushed peanuts and sweet corn filling (1 thick slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 937, "name": "Ais kacang (ABC)", "desc": "Malaysian ais kacang: a mound of shaved ice with colourful syrups, red beans, sweet corn and jelly (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 938, "name": "Oden", "desc": "Japanese Oden (4 pieces with soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 939, "name": "Bun, kaya or red bean", "desc": "green kaya coconut jam, with bun, or red bean (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 940, "name": "Chicken nuggets", "desc": "Chicken nuggets (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 941, "name": "Keropok lekor", "desc": "keropok lekor: grey-brown fried fish sausage sticks with chilli sauce (5 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 942, "name": "Rojak buah", "desc": "Malaysian buah", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 943, "name": "Teh O ais kurang manis", "desc": "a cup of black tea, with ais kurang manis (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 944, "name": "Teh C kosong", "desc": "a cup of Malaysian milk tea with evaporated milk, with kosong (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 945, "name": "Kopi (condensed milk)", "desc": "a kopitiam cup of strong Malaysian coffee (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 946, "name": "Kopi O kosong", "desc": "a kopitiam cup of strong Malaysian coffee, with o kosong (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 948, "name": "Milo kosong", "desc": "a glass of chocolate malt drink, with kosong (1 cup, no condensed milk)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 949, "name": "Limau ais kurang manis", "desc": "a glass of iced lime juice, with ais kurang manis (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 950, "name": "Sirap bandung", "desc": "a glass of pink rose syrup milk drink with ice (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 951, "name": "Milk tea, 0% sugar, no pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with 0% sugar, no", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 952, "name": "Jasmine green milk tea, less sugar", "desc": "a plastic cup of milk tea, with jasmine green , less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 953, "name": "Americano", "desc": "Americano", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 954, "name": "Latte", "desc": "Latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 955, "name": "Spanish latte", "desc": "Spanish latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 956, "name": "Fresh coconut water", "desc": "a fresh green coconut with a straw, with fresh (1 coconut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 957, "name": "100 Plus", "desc": "a glass of fizzy drink with ice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 958, "name": "Brown rice", "desc": "Brown rice (1 cup cooked)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 959, "name": "Fried chicken", "desc": "Fried chicken (1 piece (thigh))", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 960, "name": "Grilled chicken breast", "desc": "Grilled chicken breast", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 961, "name": "Tempeh, fried", "desc": "fried tempeh slices, with fried (4 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 962, "name": "Tofu, braised", "desc": "Chinese Tofu, braised (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 963, "name": "Siew mai", "desc": "three open-topped siew mai pork and shrimp dumplings with orange roe in a bamboo steamer (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 964, "name": "Dhal curry", "desc": "small bowl of yellow lentil dhal curry, with curry", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 965, "name": "Nasi lemak ayam goreng (Village Park)", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden; Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 plate with a fried chicken leg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 966, "name": "Lontong (Village Park)", "desc": "lontong: rice cakes in yellow coconut vegetable gravy with egg and sambal (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 967, "name": "Economy rice (Home Kitchen)", "desc": "Malaysian economy rice: white rice with a piece of meat and two heaps of stir-fried vegetables (Rice, 1 meat, 2 veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 968, "name": "Chicken chop (Steven's Corner)", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw (Deep-fried, with fries, coleslaw and egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 969, "name": "Banana leaf rice (Kanna Curry House)", "desc": "South Indian banana leaf rice: white rice on a banana leaf with three small heaps of vegetable curries, a papadam and curry gravy, with rice (Rice, 3 vegetables, chicken curry)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 970, "name": "Roasted chicken rice (Choon Yien)", "desc": "sliced boneless roasted chicken with glossy golden-brown skin laid in strips over cucumber, a dome of rice beside it and a small dish of chilli sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 971, "name": "Fried chicken with curry rice (Lim Fried Chicken)", "desc": "Malaysian Fried chicken with curry rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 972, "name": "Hainan chicken rice (Hometown Hainan Coffee)", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with hainan (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 973, "name": "Nasi lemak ayam goreng (Nasi Lemak Bumbung)", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden; Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 plate with a fried egg)", "neg": "whole roast chicken", "n": 1}, {"id": 974, "name": "Wantan mee (PJ Old Town)", "desc": "Malaysian wantan mee: thin springy egg noodles tossed in dark soy sauce with slices of red-edged char siu, choy sum and small boiled wontons (1 plate, dry)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 975, "name": "Yong tau foo (PJ Old Town)", "desc": "Malaysian yong tau foo: stuffed tofu, fish balls, stuffed chillies and stuffed bitter gourd in clear soup (6 pieces in soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 976, "name": "Chee cheong fun (SS2 morning market)", "desc": "plain rolled white rice noodle rolls drizzled with sweet brown sauce and sesame seeds (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 977, "name": "Claypot chicken rice (Uncle Mani's)", "desc": "a black clay pot of rice topped with chicken pieces, Chinese sausage slices and spring onion (1 pot, chicken satti soru)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 978, "name": "Hainanese chicken chop (Yut Kee)", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw, with hainanese (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 979, "name": "Hailam mee (Yut Kee)", "desc": "Hainanese fried noodles in dark sauce with pork and greens (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 980, "name": "Beef noodles (Soong Kee)", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy (1 bowl, dry noodles with soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 981, "name": "Fish porridge (Hon Kee)", "desc": "smooth Cantonese rice porridge, with fish (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 982, "name": "Banana leaf rice (Sri Nirwana Maju)", "desc": "South Indian banana leaf rice: white rice on a banana leaf with three small heaps of vegetable curries, a papadam and curry gravy, with rice (Rice, 3 vegetables, fried fish)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 983, "name": "Nasi bojari (Madam Kwan's)", "desc": "Malaysian Nasi bojari (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 984, "name": "Poached chicken rice (Chicken Rice Hub)", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with poached (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 985, "name": "Karaage don (Hungry Don)", "desc": "Japanese Karaage don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 986, "name": "Food court meal (Rock Cafe)", "desc": "Chinese Food court meal (Noodle soup, or economy rice with 2 veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 987, "name": "Burger (myBurgerLab)", "desc": "Burger (1 single burger, no fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 988, "name": "Japanese set (Hana Dining)", "desc": "Japanese set (Sashimi or grilled fish set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 989, "name": "Spring rolls, fried", "desc": "golden crispy fried spring rolls, with s, fried (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 990, "name": "Chicken curry with rice", "desc": "Malaysian chicken curry with potatoes in orange gravy, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 992, "name": "French toast", "desc": "French toast (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 993, "name": "Club sandwich", "desc": "Club sandwich (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 994, "name": "Tiramisu", "desc": "Tiramisu (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 995, "name": "Lasagna", "desc": "Lasagna (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 996, "name": "Macaroni and cheese", "desc": "Macaroni and cheese (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 997, "name": "Samosa", "desc": "Indian Samosa (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 998, "name": "Cupcake", "desc": "Cupcake (1 cupcake)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 999, "name": "Frozen yogurt", "desc": "Frozen yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1000, "name": "Chocolate sundae", "desc": "Chocolate sundae (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1001, "name": "Tacos", "desc": "Tacos (2 tacos)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1002, "name": "Macarons", "desc": "Macarons (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1003, "name": "Fried calamari", "desc": "Fried calamari (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1004, "name": "Eggs Benedict", "desc": "Eggs Benedict (2 eggs on a muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1005, "name": "Masala thosai", "desc": "a thin crispy golden dosa folded on a plate with coconut chutney and sambar, with masala (1 piece with chutney)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1006, "name": "Yakult Ace", "desc": "a small bottle of cultured milk drink", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1007, "name": "Yakult Ace Light", "desc": "a small bottle of cultured milk drink", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1008, "name": "Vitagen", "desc": "a small bottle of cultured milk drink", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1009, "name": "Vitagen Less Sugar", "desc": "a small bottle of cultured milk drink", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1010, "name": "Milo, packet drink", "desc": "a glass of chocolate malt drink, with packet drink", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1011, "name": "Chocolate milk, packet", "desc": "Chocolate milk, packet", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1012, "name": "Whey protein shake, with water", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1013, "name": "Whey protein shake, with low-fat milk", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1014, "name": "Gold Standard Whey shake, with water", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1015, "name": "Herbalife Formula 1 shake", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1016, "name": "Ensure Gold", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1017, "name": "Anlene Gold 5X", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1018, "name": "Dutch Lady +Protein milk", "desc": "Dutch Lady +Protein milk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1019, "name": "Barebells Protein Milkshake", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1020, "name": "Soy milk, sweetened packet", "desc": "a glass of soy milk, with sweetened packet", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1021, "name": "Yogurt drink", "desc": "Yogurt drink", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1022, "name": "Flavoured yogurt cup", "desc": "Flavoured yogurt cup", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1023, "name": "Iced lemon tea, packet", "desc": "Iced lemon tea, packet", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1024, "name": "Teh ais", "desc": "a tall glass of iced milk tea (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1025, "name": "Kopi ais", "desc": "a kopitiam cup of strong Malaysian coffee (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1026, "name": "Cham ais", "desc": "a cup of coffee and tea mixed with milk (1 glass, coffee and tea mixed)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1027, "name": "Neslo ais", "desc": "Neslo ais (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1028, "name": "Horlicks ais", "desc": "Horlicks ais (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1029, "name": "Teh O limau ais", "desc": "a cup of black tea; a glass of iced lime juice (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1030, "name": "Barley drink, sweetened", "desc": "a glass of barley drink with ice, with drink, sweetened (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1031, "name": "Soya cincau", "desc": "a glass of soy milk with black grass jelly (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1032, "name": "Air mata kucing", "desc": "Air mata kucing (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1033, "name": "Sugarcane juice", "desc": "a cup of pale green sugarcane juice, with juice (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1034, "name": "Watermelon juice, no sugar", "desc": "Watermelon juice, no sugar (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1035, "name": "Masala tea", "desc": "Masala tea (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1036, "name": "Brown sugar pearl fresh milk", "desc": "Brown sugar pearl fresh milk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1037, "name": "Taro milk tea with pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with taro", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1038, "name": "Fruit tea, less sugar", "desc": "Fruit tea, less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1039, "name": "Yakult green tea", "desc": "a small bottle of cultured milk drink", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1040, "name": "Jasmine green milk tea, large, normal sugar", "desc": "a plastic cup of milk tea, with jasmine green , large, normal sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1041, "name": "Da Hong Pao milk tea, less sugar", "desc": "a plastic cup of milk tea, with da hong pao , less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1042, "name": "Peach oolong milk tea, less sugar", "desc": "a plastic cup of milk tea, with peach oolong , less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1043, "name": "Matcha latte, iced", "desc": "Matcha latte, iced", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1044, "name": "Iced chocolate", "desc": "Iced chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1045, "name": "Caramel frappé", "desc": "Caramel frappé (1 tall)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1046, "name": "Vanilla milkshake", "desc": "Vanilla milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1047, "name": "All Berry Bang smoothie", "desc": "All Berry Bang smoothie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1048, "name": "Gym Junkie protein smoothie", "desc": "Gym Junkie protein smoothie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1049, "name": "llaollao, small tub with 1 topping", "desc": "small tub with 1 topping", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1050, "name": "llaollao, medium tub with 3 toppings", "desc": "medium tub with 3 toppings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1051, "name": "llaollao Sanum", "desc": "Sanum (1 cup with fruit, crunch and sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1052, "name": "Dubai chocolate strawberry cup", "desc": "Dubai chocolate strawberry cup (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1053, "name": "Inside Scoop, single scoop", "desc": "single scoop (1 scoop in a cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1054, "name": "Ice cream waffle", "desc": "Ice cream waffle (1 waffle, 1 scoop and sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1055, "name": "Soufflé pancakes", "desc": "Soufflé pancakes (2 pancakes with cream and syrup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1056, "name": "Mango bingsu", "desc": "Korean Mango bingsu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1057, "name": "Chocolate bingsu", "desc": "Korean Chocolate bingsu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1058, "name": "Mango sago pomelo", "desc": "Chinese Mango sago pomelo (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1059, "name": "Red bean soup", "desc": "Chinese Red bean soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1060, "name": "Bubur cha cha", "desc": "bubur cha cha: coconut milk dessert with sweet potato and taro cubes (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1061, "name": "Sago gula melaka", "desc": "white sago pudding with dark palm sugar syrup and coconut milk (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1062, "name": "Mango sticky rice", "desc": "Thai Mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1063, "name": "Muah chee", "desc": "muah chee: chewy glutinous rice dough pieces coated in crushed peanuts and sugar", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1064, "name": "Mochi", "desc": "Japanese Mochi (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1065, "name": "Kaya puff", "desc": "green kaya coconut jam, with puff (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1066, "name": "Polo bun with butter", "desc": "Hong Kong pineapple bun with crackly golden top and a slab of butter, with butter (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1067, "name": "Chicken floss bun", "desc": "Chinese Chicken floss bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1068, "name": "Brownie", "desc": "Brownie (1 square)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1069, "name": "Churros with chocolate dip", "desc": "Churros with chocolate dip (4 sticks)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1070, "name": "Crêpe with banana and chocolate", "desc": "Crêpe with banana and chocolate (1 crêpe)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1071, "name": "Molten lava cake with ice cream", "desc": "Molten lava cake with ice cream (1 cake, 1 scoop)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1072, "name": "McFlurry with Oreo", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1073, "name": "Ais krim potong", "desc": "Malaysian Ais krim potong (1 stick)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1074, "name": "Chocolate-coated ice cream stick", "desc": "Chocolate-coated ice cream stick (1 stick)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1075, "name": "Ice cream cone, packaged", "desc": "Ice cream cone, packaged (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1076, "name": "Corn snack pack", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1077, "name": "Seaweed snack", "desc": "crispy seaweed snack sheets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1078, "name": "Granola bar", "desc": "Granola bar (1 bar)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1079, "name": "Popcorn, salted", "desc": "Popcorn, salted", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1080, "name": "Chicken siew mai", "desc": "three open-topped siew mai pork and shrimp dumplings with orange roe in a bamboo steamer, with chicken (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1081, "name": "Steamed fish balls", "desc": "white round fish balls, with steamed s (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1082, "name": "Steamed pork ribs with black bean", "desc": "Chinese Steamed pork ribs with black bean", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1083, "name": "Chicken feet in black bean sauce", "desc": "Chinese Chicken feet in black bean sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1084, "name": "Beancurd skin roll, fried", "desc": "Chinese Beancurd skin roll, fried (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1085, "name": "Prawn cheong fun", "desc": "rolled smooth white rice noodle rolls drizzled with sweet soy sauce and sesame, with prawn (1 plate, 3 rolls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1086, "name": "Char siu cheong fun", "desc": "rolled smooth white rice noodle rolls drizzled with sweet soy sauce and sesame; slices of red-edged glossy char siu barbecued pork (1 plate, 3 rolls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1087, "name": "Lo bak go, pan-fried radish cake", "desc": "pan-fried golden radish cake squares, with pan-fried radish cake (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1088, "name": "Yam puff (wu kok)", "desc": "golden lacy crispy deep-fried yam puffs (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1089, "name": "Ham sui kok", "desc": "oval glutinous rice dumplings, golden and fried (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1090, "name": "Fried prawn dumplings with mayo", "desc": "Chinese pleated dumplings, with fried prawn with mayo (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 1092, "name": "Sui kow in soup", "desc": "large sui kow prawn dumplings in clear soup, with in soup (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1093, "name": "Sheng jian bao", "desc": "pan-fried pork buns with crispy golden bottoms and sesame (3 pieces, pan-fried)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1094, "name": "Salted egg custard bun (lau sar bao)", "desc": "Chinese Salted egg custard bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1095, "name": "Egg custard bun (nai wong bao)", "desc": "Chinese Egg custard bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1096, "name": "Lotus paste bun", "desc": "Chinese Lotus paste bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1097, "name": "Chicken bao", "desc": "fluffy white steamed bun, with chicken (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1098, "name": "Big bao (dai bao)", "desc": "fluffy white steamed bun", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1099, "name": "Fried mantou with condensed milk", "desc": "Chinese Fried mantou with condensed milk (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1100, "name": "Ma lai gao", "desc": "a slice of fluffy brown steamed sponge cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1101, "name": "Sesame balls (jin deui)", "desc": "golden fried sesame balls covered in white sesame seeds (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1102, "name": "Char siu sou", "desc": "slices of red-edged glossy char siu barbecued pork (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1103, "name": "Century egg and pork porridge", "desc": "smooth Cantonese rice porridge, with century egg and pork (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1104, "name": "Stuffed tofu and brinjal", "desc": "Chinese Stuffed tofu and brinjal (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1105, "name": "Bak chang", "desc": "a pyramid-shaped glutinous rice dumpling wrapped in bamboo leaves, opened to show savoury filling (1 rice dumpling)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1106, "name": "Pain au chocolat", "desc": "Pain au chocolat (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1107, "name": "Crookie", "desc": "Crookie (1 croissant with cookie dough)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1108, "name": "Chicken ham and cheese croissant", "desc": "Chicken ham and cheese croissant (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1109, "name": "Fruit Danish", "desc": "Fruit Danish (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1110, "name": "Cinnamon roll", "desc": "Cinnamon roll (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1111, "name": "Blueberry muffin", "desc": "Blueberry muffin (1 muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1112, "name": "Chocolate muffin", "desc": "Chocolate muffin (1 muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1113, "name": "Banana bread", "desc": "Banana bread (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1114, "name": "Cream puff", "desc": "Cream puff (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1115, "name": "Chocolate éclair", "desc": "Chocolate éclair (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1116, "name": "Swiss roll", "desc": "Swiss roll (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1117, "name": "Pandan chiffon cake", "desc": "green pandan, with chiffon cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1118, "name": "Red velvet cake", "desc": "Red velvet cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1119, "name": "Carrot cake with cream cheese", "desc": "Carrot cake with cream cheese (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1120, "name": "Mille crêpe", "desc": "Mille crêpe (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1121, "name": "Portuguese egg tart", "desc": "Portuguese egg tart with caramelised burnt-spotted custard top (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1122, "name": "Mooncake, lotus paste", "desc": "a mooncake cut in quarters showing lotus paste filling, with lotus paste (1 quarter)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1123, "name": "Taiyaki", "desc": "Japanese Taiyaki (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1124, "name": "Chicken sausage roll", "desc": "Chicken sausage roll (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1125, "name": "Tuna puff", "desc": "Tuna puff (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1126, "name": "Sausage bun", "desc": "Sausage bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1127, "name": "Mexican coffee bun", "desc": "Mexican coffee bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1128, "name": "Tuna mayo bun", "desc": "Tuna mayo bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1129, "name": "Korean garlic cream cheese bun", "desc": "Korean garlic cream cheese bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1130, "name": "Curry chicken bun", "desc": "Malaysian chicken curry with potatoes in orange gravy (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1131, "name": "Cheese bun", "desc": "Cheese bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1132, "name": "Pretzel, cinnamon sugar", "desc": "Pretzel, cinnamon sugar (1 pretzel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1133, "name": "White bread", "desc": "White bread (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1134, "name": "Wholemeal bread", "desc": "Wholemeal bread (2 slices)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 1135, "name": "Green bean soup", "desc": "Chinese Green bean soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1136, "name": "Black sesame paste", "desc": "Chinese Black sesame paste (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1137, "name": "Peanut paste", "desc": "Chinese Peanut paste (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1138, "name": "Walnut paste", "desc": "Chinese Walnut paste (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1139, "name": "Almond paste", "desc": "Chinese Almond paste (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1140, "name": "Sweet potato ginger soup", "desc": "Chinese Sweet potato ginger soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1141, "name": "Tang yuan in ginger syrup", "desc": "glutinous rice balls in ginger syrup, with in ginger syrup (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1142, "name": "Snow fungus with longan", "desc": "Chinese Snow fungus with longan (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1143, "name": "Barley, fu chuk and gingko", "desc": "a glass of barley drink with ice, with fu chuk and gingko (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1144, "name": "Bubur pulut hitam", "desc": "black glutinous rice porridge with a swirl of coconut milk (1 bowl with coconut milk)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1145, "name": "Honeydew sago", "desc": "Chinese Honeydew sago (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1146, "name": "Leng chee kang", "desc": "leng chee kang: clear sweet dessert soup with longan, lotus seeds and white fungus (1 bowl, iced)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1147, "name": "Gui ling gao herbal jelly", "desc": "Chinese Gui ling gao herbal jelly (1 bowl with syrup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1148, "name": "Taro sago", "desc": "Chinese Taro sago (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1149, "name": "Salt bread (shio pan)", "desc": "Korean Salt bread (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1150, "name": "Kunafa", "desc": "Middle Eastern Kunafa (1 slice with cheese and syrup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1151, "name": "Cromboloni", "desc": "Cromboloni (1 filled round croissant)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1152, "name": "Mochi cheesecake", "desc": "Mochi cheesecake (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1153, "name": "Tanghulu", "desc": "Chinese Tanghulu (1 skewer of candied fruit)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1154, "name": "Jagung cheese", "desc": "Malaysian Jagung cheese (1 cup of corn with cheese)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1155, "name": "Fruit sando", "desc": "Japanese Fruit sando (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1156, "name": "Musang King durian", "desc": "Musang King durian", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1157, "name": "Durian cendol", "desc": "Malaysian cendol: shaved ice with green rice-flour jelly strands, red beans, coconut milk and dark palm sugar syrup, with durian (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1158, "name": "Pandan gula melaka cake", "desc": "green pandan, with gula melaka cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1159, "name": "Garrett popcorn, Chicago mix", "desc": "Chicago mix", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1160, "name": "Mixue lemonade", "desc": "lemonade", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1161, "name": "Brown sugar boba milk (Tiger Sugar)", "desc": "with black tapioca pearls at the bottom and a thick straw, with brown sugar milk (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1162, "name": "Strawberry matcha latte", "desc": "Strawberry matcha latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1163, "name": "Pistachio latte", "desc": "Pistachio latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1164, "name": "Einspänner cream coffee", "desc": "Einspänner cream coffee (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1165, "name": "Coconut shake", "desc": "Malaysian shake (1 cup with ice cream)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1166, "name": "Soda limau asam boi", "desc": "a glass of iced lime juice, with soda asam boi (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1167, "name": "Air katira", "desc": "Malaysian Air katira (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 2001, "name": "Pizza, margherita", "desc": "a whole round margherita pizza cut into eight slices. A puffy golden-brown baked crust with darker toasted spots, red tomato sauce, many round pools of melted white mozzarella with browned patches, and plenty of fresh green basil leaves", "neg": "", "n": 1}, {"id": 2002, "name": "Pizza, pepperoni", "desc": "a whole round pepperoni pizza cut into eight slices. A puffy golden-brown baked crust with darker toasted spots, melted bubbly cheese with browned patches, fully covered edge to edge with many round red pepperoni slices that are slightly curled and crisp at the edges", "neg": "", "n": 1}, {"id": 2003, "name": "Pizza, Hawaiian chicken", "desc": "a whole round Hawaiian pizza cut into eight slices. A puffy golden-brown baked crust with darker toasted spots, melted bubbly cheese with browned patches, generously covered with many yellow pineapple chunks and pink pieces of chicken ham", "neg": "", "n": 1}, {"id": 2004, "name": "Pizza, meat lovers", "desc": "a whole round meat lovers pizza cut into eight slices. A puffy golden-brown baked crust, melted bubbly cheese, heaped with pepperoni slices, brown minced beef crumbles, sliced sausage and pieces of chicken", "neg": "", "n": 1}, {"id": 2005, "name": "Pizza, veggie", "desc": "a whole round vegetable pizza cut into eight slices. A puffy golden-brown baked crust, melted bubbly cheese, generously covered with green capsicum rings, sliced mushrooms, black olives, red onion rings, sweetcorn and tomato slices", "neg": "", "n": 1}, {"id": 2006, "name": "Pizza, BBQ chicken", "desc": "a whole round BBQ chicken pizza cut into eight slices. A puffy golden-brown baked crust, melted bubbly cheese, generously covered with chunks of grilled chicken, red onion rings and green capsicum, with dark brown barbecue sauce drizzled in zigzags", "neg": "", "n": 1}, {"id": 2007, "name": "Curry puff, mini", "desc": "a small plate with three mini Malaysian curry puffs. Three small half-moon golden fried pastries with neatly crimped braided edges, one broken open to show yellow curry potato filling", "neg": "", "n": 1}, {"id": 3001, "name": "Pork noodle soup", "desc": "a bowl of Malaysian pork noodle soup. Thick round white rice noodles in cloudy light brown pork broth, with pinkish crumbly minced pork, a few slices of dark brown pork liver, pieces of pork intestine, lean pork slices and bright green choy sum and dark green sweet potato leaves", "neg": "", "n": 1}, {"id": 3002, "name": "Pork noodle soup, dry", "desc": "a bowl of dry pork noodles: rice noodles tossed in dark soy sauce with crispy pork lard bits, topped with minced pork, liver slices and greens, with a small separate bowl of pork soup beside it", "neg": "", "n": 1}, {"id": 3003, "name": "Wantan mee, soup", "desc": "a bowl of Malaysian wantan mee soup. Thin springy yellow egg noodles in clear light broth, with four plump wantan dumplings with thin wrinkled wrappers, slices of red-edged char siu and green choy sum", "neg": "", "n": 1}, {"id": 3004, "name": "Beef ball noodle soup", "desc": "a bowl of beef ball noodle soup. Flat white rice noodles in clear beef broth with six round brown beef balls, thin slices of beef, bean sprouts, chopped spring onion and coriander", "neg": "", "n": 1}, {"id": 3005, "name": "Beef ball noodles, dry", "desc": "a bowl of dry beef noodles: noodles coated in dark sauce topped with brown minced beef and chopped spring onion, with a small separate bowl of clear soup holding round beef balls", "neg": "", "n": 1}, {"id": 3006, "name": "Beef noodle soup", "desc": "a big bowl of beef noodle soup. Wheat noodles in rich dark brown beef broth with large chunks of braised beef, bok choy and chopped spring onion", "neg": "", "n": 1}, {"id": 3007, "name": "Fish ball noodle soup", "desc": "a bowl of fish ball noodle soup. Thin yellow noodles in clear broth with round white fish balls, slices of fish cake with a pink edge, lettuce and fried shallots", "neg": "", "n": 1}, {"id": 3008, "name": "Fish head noodle, milky soup", "desc": "a bowl of fish head noodle soup. Thick white rice noodles in milky white fish broth with golden fried fish pieces, tomato wedges, salted mustard greens and chopped spring onion", "neg": "", "n": 1}, {"id": 3009, "name": "Fish slice noodle soup", "desc": "a bowl of sliced fish noodle soup. White rice noodles in clear broth with soft white slices of poached fish, tomato wedges, ginger strips and green vegetables", "neg": "", "n": 1}, {"id": 3010, "name": "Kway teow soup", "desc": "a bowl of kway teow soup. Flat wide white rice noodles in clear broth with fish balls, thin pork slices, bean sprouts, chopped spring onion and fried garlic", "neg": "", "n": 1}, {"id": 3011, "name": "Duck noodle soup", "desc": "a bowl of duck noodle soup. Yellow noodles in dark herbal broth with slices of braised duck with brown skin, green vegetables and coriander", "neg": "", "n": 1}, {"id": 3012, "name": "Duck noodles, dry", "desc": "a plate of dry duck noodles: yellow noodles in dark glossy sauce topped with slices of braised duck with brown skin and cucumber, with a small bowl of soup beside it", "neg": "", "n": 1}, {"id": 3013, "name": "Chicken noodle soup", "desc": "a bowl of chicken noodle soup. Thin yellow noodles in clear golden chicken broth with shredded white chicken, green choy sum and fried shallots", "neg": "", "n": 1}, {"id": 3014, "name": "Chicken noodles, dry", "desc": "a plate of dry chicken noodles: noodles tossed in light soy and sesame oil, topped with neat slices of pale poached chicken and choy sum, with a small bowl of soup beside it", "neg": "", "n": 1}, {"id": 3015, "name": "Mee sua soup with chicken", "desc": "a bowl of Foochow mee sua: very fine thin white wheat vermicelli in reddish red wine chicken soup with chicken pieces, a hard boiled egg and shredded black fungus", "neg": "", "n": 1}, {"id": 3016, "name": "Sang har mee", "desc": "a plate of sang har mee: a nest of golden crispy fried egg noodles under glossy pale yellow egg gravy, topped with a big freshwater prawn cut in half lengthwise showing orange roe, with green vegetables", "neg": "", "n": 1}, {"id": 3017, "name": "Cantonese fried noodles (yee mee)", "desc": "a plate of Cantonese fried noodles: golden crispy fried noodles covered with thick glossy egg gravy with pork slices, prawns, fish cake slices and green choy sum", "neg": "", "n": 1}, {"id": 3018, "name": "Curry mee", "desc": "a bowl of Malaysian curry mee. Yellow noodles and rice vermicelli in orange-red coconut curry broth with tofu puffs, chicken pieces, long beans, cockles, a dollop of red sambal and a slick of red chilli oil on top", "neg": "", "n": 1}, {"id": 3019, "name": "Mee soto", "desc": "a bowl of mee soto. Yellow noodles in clear yellowish spiced chicken soup with shredded chicken, bean sprouts, fried shallots, chopped celery and a small dish of sambal kicap", "neg": "", "n": 1}, {"id": 3020, "name": "Mee hoon kueh soup", "desc": "a bowl of mee hoon kueh: hand-torn flat irregular flour dough pieces in clear broth with minced pork, crispy ikan bilis, sliced mushroom and dark green sweet potato leaves", "neg": "", "n": 1}, {"id": 3021, "name": "Mee jawa", "desc": "a plate of mee jawa: yellow noodles under thick orange sweet potato and prawn gravy, topped with fried tofu, half a boiled egg, bean sprouts, prawn fritter pieces and a lime", "neg": "", "n": 1}, {"id": 3022, "name": "Claypot loh shu fun", "desc": "a black claypot of loh shu fun: short fat white rice noodles in dark soy sauce with minced pork, a raw egg yolk on top, chopped spring onion", "neg": "", "n": 1}, {"id": 3023, "name": "Kon lo mee", "desc": "a plate of Malaysian kon lo mee: thin springy egg noodles tossed in light soy sauce and lard, topped with slices of red-edged char siu and green choy sum", "neg": "", "n": 1}, {"id": 3024, "name": "Kampua mee", "desc": "a plate of Sarawak kampua mee: thin springy curly noodles glossy with shallot oil, topped with thin slices of red char siu, minced pork and chopped spring onion", "neg": "", "n": 1}, {"id": 3025, "name": "Hakka mee", "desc": "a plate of Hakka mee: plain springy noodles topped with brown minced pork sauce and shredded lettuce", "neg": "", "n": 1}, {"id": 3026, "name": "Bak chor mee", "desc": "a bowl of bak chor mee: flat yellow noodles tossed in black vinegar and chilli sauce, topped with minced pork, slices of pork liver, sliced mushrooms, fried lard and a fish ball", "neg": "", "n": 1}, {"id": 3027, "name": "Mee kicap", "desc": "a plate of mee kicap: yellow noodles in dark sweet soy sauce with pieces of chicken and green vegetables", "neg": "", "n": 1}, {"id": 3028, "name": "Kway teow goreng", "desc": "a plate of mamak kway teow goreng: flat rice noodles stir-fried with red chilli paste and egg, with chicken pieces, bean sprouts and green leaves, and a lime", "neg": "", "n": 1}, {"id": 3029, "name": "Bee hoon goreng", "desc": "a plate of fried bee hoon: thin white rice vermicelli stir-fried light brown with egg, chicken pieces, shredded cabbage and carrot, topped with fried shallots", "neg": "", "n": 1}, {"id": 3030, "name": "Bee hoon soup", "desc": "a bowl of bee hoon soup: thin white rice vermicelli in clear broth with fish balls, fish cake slices and green vegetables", "neg": "", "n": 1}, {"id": 3031, "name": "Zha jiang mian", "desc": "a bowl of zha jiang mian: thick white wheat noodles topped with dark brown minced pork and fermented bean sauce and julienned cucumber", "neg": "", "n": 1}, {"id": 3032, "name": "Dan dan noodles", "desc": "a bowl of dan dan noodles: wheat noodles in a pool of red chilli oil and sesame sauce, topped with crispy minced pork, crushed peanuts and bok choy", "neg": "", "n": 1}, {"id": 3033, "name": "Mala noodles", "desc": "a bowl of mala noodles: noodles in bright red spicy oily broth with dried red chillies and Sichuan peppercorns, sliced beef, tofu skin and green vegetables", "neg": "", "n": 1}, {"id": 3034, "name": "Glass noodle stir-fry", "desc": "a plate of stir-fried glass noodles: clear glossy mung bean noodles stir-fried with egg, cabbage, carrot strips and spring onion", "neg": "", "n": 1}, {"id": 3035, "name": "Tomato egg noodle soup", "desc": "a bowl of tomato and egg noodle soup: noodles in red-orange tomato broth with soft yellow egg ribbons, tomato chunks and chopped spring onion", "neg": "", "n": 1}, {"id": 4001, "name": "White rice (economy rice)", "desc": "a small round mound of fluffy plain white steamed rice on a small white plate", "neg": "", "n": 1}, {"id": 4002, "name": "Brown rice (economy rice)", "desc": "a small round mound of light brown wholegrain rice on a small white plate", "neg": "", "n": 1}, {"id": 4003, "name": "Oil rice (chicken rice)", "desc": "a small round dome of glossy pale yellow fragrant chicken rice on a small white plate, with a pandan leaf", "neg": "", "n": 1}, {"id": 4004, "name": "Nasi lemak rice", "desc": "a small round mound of white coconut rice on a piece of green banana leaf, with a pandan knot on top", "neg": "", "n": 1}, {"id": 4005, "name": "Basmati rice, plain", "desc": "a small mound of long separate grains of white basmati rice on a small white plate", "neg": "", "n": 1}, {"id": 4006, "name": "Briyani rice", "desc": "a small mound of long-grain briyani rice with yellow and orange grains, fried onions and a few raisins and cashews on a small white plate", "neg": "", "n": 1}, {"id": 4007, "name": "Turmeric rice (nasi kunyit)", "desc": "a small mound of bright golden yellow turmeric glutinous rice on a small white plate", "neg": "", "n": 1}, {"id": 4008, "name": "Plain fried rice", "desc": "a small mound of plain egg fried rice with bits of scrambled egg and spring onion on a small white plate", "neg": "", "n": 1}, {"id": 4009, "name": "Sweet and sour pork", "desc": "a small dish of sweet and sour pork: crispy battered pork pieces in glossy bright red-orange sauce with pineapple chunks, green capsicum and onion", "neg": "", "n": 1}, {"id": 4010, "name": "Sweet and sour chicken", "desc": "a small dish of sweet and sour chicken: crispy battered chicken pieces in glossy bright red-orange sauce with pineapple chunks, green capsicum and onion", "neg": "", "n": 1}, {"id": 4011, "name": "Marmite chicken", "desc": "a small dish of marmite chicken: fried chicken pieces coated in a sticky dark brown glossy marmite glaze, sprinkled with white sesame seeds", "neg": "", "n": 1}, {"id": 4012, "name": "Kung pao chicken", "desc": "a small dish of kung pao chicken: diced chicken in glossy dark brown sauce with dried red chillies, roasted peanuts and spring onion", "neg": "", "n": 1}, {"id": 4013, "name": "Black pepper chicken", "desc": "a small dish of black pepper chicken: pieces of chicken in thick glossy brown black pepper sauce with sliced onion and green capsicum", "neg": "", "n": 1}, {"id": 4014, "name": "Lemon chicken", "desc": "a small dish of lemon chicken: golden crispy battered chicken slices topped with glossy yellow lemon sauce and a lemon slice", "neg": "", "n": 1}, {"id": 4015, "name": "Curry chicken (economy rice)", "desc": "a small bowl of curry chicken: a piece of chicken and a potato chunk in orange-red coconut curry gravy", "neg": "", "n": 1}, {"id": 4016, "name": "Braised pork belly (tau yu bak)", "desc": "a small bowl of tau yu bak: slices of pork belly with fat layers and a halved hard-boiled egg braised in dark brown soy sauce", "neg": "", "n": 1}, {"id": 4017, "name": "Fried chicken wing (economy rice)", "desc": "a single crispy golden-brown fried chicken wing on a small white plate", "neg": "", "n": 1}, {"id": 4018, "name": "Fried fish fillet", "desc": "a crispy golden battered fried fish fillet on a small white plate", "neg": "", "n": 1}, {"id": 4019, "name": "Steamed fish (economy rice)", "desc": "a piece of steamed white fish with pale skin in light soy sauce topped with ginger strips and spring onion on a small white plate", "neg": "", "n": 1}, {"id": 4020, "name": "Sambal fish", "desc": "a piece of fried fish covered in red sambal chilli paste with sliced onion on a small white plate", "neg": "", "n": 1}, {"id": 4021, "name": "Sambal prawns", "desc": "a small dish of prawns in red sambal chilli paste with sliced onion", "neg": "", "n": 1}, {"id": 4022, "name": "Luncheon meat slices", "desc": "two pan-fried slices of pink luncheon meat with golden browned edges on a small white plate", "neg": "", "n": 1}, {"id": 4023, "name": "Egg foo yong (omelette)", "desc": "a round golden fluffy Chinese omelette with bits of onion and carrot on a small white plate", "neg": "", "n": 1}, {"id": 4024, "name": "Sambal egg", "desc": "a hard-boiled egg cut in half, covered in red sambal chilli sauce, on a small white plate", "neg": "", "n": 1}, {"id": 4025, "name": "Braised tofu (economy rice)", "desc": "a small dish of golden fried tofu cubes braised in glossy brown oyster sauce with mushrooms", "neg": "", "n": 1}, {"id": 4026, "name": "Mapo tofu", "desc": "a small dish of mapo tofu: soft white tofu cubes in red spicy chilli bean sauce with minced pork and spring onion", "neg": "", "n": 1}, {"id": 4027, "name": "Stir-fried spinach (bayam)", "desc": "a small dish of stir-fried dark green spinach leaves with garlic", "neg": "", "n": 1}, {"id": 4028, "name": "Kangkung belacan", "desc": "a small dish of stir-fried kangkung water spinach with hollow stems and long leaves in reddish belacan chilli paste", "neg": "", "n": 1}, {"id": 4029, "name": "Stir-fried cabbage", "desc": "a small dish of stir-fried pale green cabbage with carrot strips", "neg": "", "n": 1}, {"id": 4030, "name": "Lettuce with oyster sauce", "desc": "a small dish of blanched bright green lettuce leaves drizzled with brown oyster sauce and fried garlic", "neg": "", "n": 1}, {"id": 4031, "name": "Long beans with egg", "desc": "a small dish of chopped green long beans stir-fried with bits of yellow scrambled egg", "neg": "", "n": 1}, {"id": 4032, "name": "Brinjal with sambal", "desc": "a small dish of soft purple brinjal (eggplant) slices topped with red sambal", "neg": "", "n": 1}, {"id": 4033, "name": "Bean sprouts stir-fry", "desc": "a small dish of stir-fried white bean sprouts with chives", "neg": "", "n": 1}, {"id": 4034, "name": "Curry vegetables (sayur lodeh)", "desc": "a small bowl of sayur lodeh: cabbage, long beans, carrot and tofu in yellow coconut curry", "neg": "", "n": 1}, {"id": 4035, "name": "Stir-fried broccoli", "desc": "a small dish of stir-fried green broccoli florets with carrot slices", "neg": "", "n": 1}, {"id": 4036, "name": "Bitter gourd with egg", "desc": "a small dish of sliced green bitter gourd stir-fried with yellow scrambled egg", "neg": "", "n": 1}, {"id": 4037, "name": "Stir-fried choy sum", "desc": "a small dish of stir-fried green choy sum with garlic", "neg": "", "n": 1}, {"id": 4038, "name": "Curry gravy on rice", "desc": "a ladle of orange-red curry gravy poured over a small mound of white rice", "neg": "", "n": 1}]
STYLES = {
 "Sticker": "a cute hand-drawn food sticker illustration of {d}. Chubby rounded soft shapes, pastel colours with soft pink accents, smooth clean outlines, simple soft shading, tiny white shine highlights, like a sticker from a Japanese stationery set, just the food on its dish, three-quarter view, centered, plain white background"
}

JOB = [p for p in PROMPTS if p["id"] in set(ONLY)]


In [ ]:
import json, os, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
job = JOB
json.dump({"style": STYLES[STYLE_KEY], "size": SIZE, "steps": STEPS, "items": job}, open("/kaggle/working/job.json", "w"))
open("/kaggle/working/worker.py", "w").write(r"""
import sys, json, time, torch, numpy as np
from PIL import Image
gpu, n = int(sys.argv[1]), int(sys.argv[2])
import os
J = json.load(open("/kaggle/working/job.json")); items = [p for p in J["items"][gpu::n] if not os.path.exists(f"/kaggle/working/c{p['id']}.webp")]
from diffusers import ZImagePipeline, ZImageTransformer2DModel, BitsAndBytesConfig as DB
from transformers import BitsAndBytesConfig as TB, AutoModel
from rembg import remove, new_session
R = "Tongyi-MAI/Z-Image-Turbo"; dev = f"cuda:{gpu}"
MODE = ""
def load(tdt, vae32, small_te=True):
    # tdt: number format for the painting part. vae32: run the final picture decoder in full precision.
    tr = ZImageTransformer2DModel.from_pretrained(R, subfolder="transformer", torch_dtype=tdt,
        quantization_config=DB(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=tdt), device_map={"": dev})
    kw = {}
    if small_te:
        try:
            kw["text_encoder"] = AutoModel.from_pretrained(R, subfolder="text_encoder", torch_dtype=tdt,
                quantization_config=TB(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=tdt), device_map={"": dev})
        except Exception as e: print("4-bit text reader not available:", str(e)[:200], flush=True)
    p = ZImagePipeline.from_pretrained(R, transformer=tr, torch_dtype=tdt, **kw)
    if "text_encoder" in kw: p.vae.to(dev)
    else: p.enable_model_cpu_offload(gpu_id=gpu)
    if vae32: p.vae.to(dtype=torch.float32)
    return p
def gen(prompt, seed):
    g = torch.Generator("cpu").manual_seed(seed)
    return pipe(prompt=prompt, height=J["size"], width=J["size"], num_inference_steps=J["steps"], guidance_scale=0.0, generator=g).images[0]
def good(im):
    a = np.asarray(im).astype("float32"); return a.mean() > 8 and a.std() > 6      # a blank or single-colour picture means the numbers overflowed
LONG = J["style"].format(d=max((p["desc"] for p in J["items"]), key=len))   # the longest real prompt, so a memory problem shows up now
# Try the fast settings first and keep the first one that draws a real picture. The last one is slow but known to work.
LADDER = [("full precision painting", torch.float32, True, True),      # measured: about 51 s a picture, works
          ("bfloat16 (slow, known to work)", torch.bfloat16, False, True),
          ("bfloat16, swapping parts in and out (slowest)", torch.bfloat16, False, False)]
import gc
pipe = None
for name, tdt, vae32, small in LADDER:
    try:
        t1 = time.time(); pipe = load(tdt, vae32, small); tl = time.time() - t1
        t1 = time.time(); test = gen(LONG, 1); tg = time.time() - t1
        ok = good(test); print(f"gpu{gpu}: tried {name}: loaded in {tl:.0f} s, test picture {tg:.0f} s, {'OK' if ok else 'blank picture'}", flush=True)
        if ok: MODE = name; break
    except Exception as e:
        print(f"gpu{gpu}: tried {name}: failed, {type(e).__name__}: {str(e)[:200]}", flush=True)
    pipe = None; gc.collect(); torch.cuda.empty_cache()
if pipe is None: print(f"gpu{gpu}: no setting worked", flush=True); sys.exit(1)
print(f"gpu{gpu}: using {MODE}; memory in use {torch.cuda.memory_allocated(gpu)/2**30:.1f} GB, peak {torch.cuda.max_memory_allocated(gpu)/2**30:.1f} GB", flush=True)
open(f"/kaggle/working/ready{gpu}", "w").write("1")
CUT = new_session("isnet-general-use"); t0 = time.time(); done = 0
for p in items:
    try:
        im = gen(J["style"].format(d=p["desc"]), 17)
        o = remove(im, session=CUT).convert("RGBA"); bb = o.getbbox()
        if bb: o = o.crop(bb)
        o.thumbnail((256, 256), Image.LANCZOS); o.save(f"/kaggle/working/c{p['id']}.webp", "WEBP", quality=80, method=6); done += 1
    except Exception as e: print("failed", p["id"], str(e)[:160], flush=True); torch.cuda.empty_cache()
    if done and done % 10 == 0: print(f"gpu{gpu}: {done}/{len(items)} done, {(time.time()-t0)/max(done,1):.1f} s each", flush=True)
print(f"gpu{gpu} finished {done} of {len(items)} in {time.time()-t0:.0f} s, {(time.time()-t0)/max(done,1):.1f} s each, {MODE}", flush=True)
""")
import time, torch, glob
n = max(1, torch.cuda.device_count()); print("GPUs found:", n, flush=True)
def start(i, k):
    log = open(f"/kaggle/working/log{i}.txt", "w")
    return subprocess.Popen([sys.executable, "/kaggle/working/worker.py", str(i), str(k)], stdout=log, stderr=subprocess.STDOUT)
# start the GPUs one after another: loading two copies of the model at once runs the machine out of memory
procs = []
for i in range(n):
    pr = start(i, n); procs.append(pr); t = time.time()
    while pr.poll() is None and not os.path.exists(f"/kaggle/working/ready{i}") and time.time() - t < 1500: time.sleep(5)
    print(f"gpu{i}: {'ready' if os.path.exists(f'/kaggle/working/ready{i}') else 'did not start'} after {time.time()-t:.0f} s", flush=True)
for i, pr in enumerate(procs):
    code = pr.wait(); print(f"gpu{i} ended with code {code}", flush=True)
    print("".join([l for l in open(f"/kaggle/working/log{i}.txt", errors="replace").read().replace("\r", "\n").splitlines(True) if "it/s]" not in l and "s/it]" not in l][-25:]), flush=True)
left = [p for p in JOB if not os.path.exists(f"/kaggle/working/c{p['id']}.webp")]
if left:
    print("finishing", len(left), "missed pictures on the first GPU", flush=True)
    pr = start(0, 1); print("retry ended with code", pr.wait(), flush=True)
    print("".join(open("/kaggle/working/log0.txt", errors="replace").read().replace("\r", "\n").splitlines(True)[-15:]), flush=True)
for f in glob.glob("/kaggle/working/ready*") + glob.glob("/kaggle/working/log*.txt"): os.remove(f)
print("all workers finished")


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import glob
try: f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
except Exception: f = ImageFont.load_default()
made = [(p, f"/kaggle/working/c{p['id']}.webp") for p in JOB if os.path.exists(f"/kaggle/working/c{p['id']}.webp")]
print("pictures made:", len(made), "of", len(JOB))
C = 200; W = 8; PER = 48
for s in range(0, len(made), PER):
    part = made[s:s + PER]; H = (len(part) + W - 1) // W
    sheet = Image.new("RGB", (W * C, H * (C + 34)), "#fff0f4"); dr = ImageDraw.Draw(sheet)
    for k, (p, fn) in enumerate(part):
        im = Image.open(fn).convert("RGBA"); x, y = (k % W) * C, (k // W) * (C + 34); im.thumbnail((C - 16, C - 16))
        sheet.paste(im, (x + (C - im.width) // 2, y + 6 + (C - 16 - im.height) // 2), im)
        dr.text((x + 6, y + C - 8), f"{p['id']} {p['name'][:26]}", fill="#7a2e4d", font=f)
    sheet.save(f"/kaggle/working/contact_dish_{s:04d}.jpg", quality=82)
os.remove("/kaggle/working/job.json"); os.remove("/kaggle/working/worker.py")
print("sheets done")